In [ ]:
from transformers import AutoConfig, AutoModel
import pandas as pd

models = [
 "bert-base-uncased",
 "google/bigbird-roberta-base",
 "distilbert-base-uncased",
 "allenai/longformer-base-4096",
 "monologg/bert-base-cased-goemotions-original"
]

results = []

for model_name in models:
 try:
 # Cargamos la configuración para ver hiperparámetros
 config = AutoConfig.from_pretrained(model_name)
 # Cargamos el modelo (solo arquitectura) para contar parámetros
 model = AutoModel.from_pretrained(model_name)
 
 # Extraer valores
 layers = getattr(config, "num_hidden_layers", "N/A")
 hidden_size = config.hidden_size
 # El MLP suele ser 4 veces el hidden_size
 intermediate_size = getattr(config, "intermediate_size", hidden_size * 4)
 total_neurons = layers * intermediate_size
 max_seq = config.max_position_embeddings
 params = sum(p.numel() for p in model.parameters()) / 1e6 # En millones

 results.append({
 "Model": model_name.split('/')[-1],
 "Layers": layers,
 "Hidden Size": hidden_size,
 "Total Neurons": f"{total_neurons:,}",
 "Max Seq": max_seq,
 "Params (M)": f"{params:.1f}M"
 })
 except Exception as _e:
 print(f"[info] skip stats for {model_name} (not cached offline): {_e.__class__.__name__}")
 continue

df = pd.DataFrame(results)
print(df.to_string(index=False))

In [ ]:
import csv
import matplotlib.pyplot as plt

class F1Tracker:
 def __init__(self, experiment_name, save_csv=True, csv_path="f1_results.csv"):
 self.experiment_name = experiment_name
 self.percent_silenced = []
 self.f1_scores = []
 self.save_csv = save_csv
 self.csv_path = csv_path
 if self.save_csv:
 with open(self.csv_path, mode='w', newline='') as file:
 writer = csv.writer(file)
 writer.writerow(["experiment", "percent_silenced", "f1_score"])

 def add(self, percent, f1_score):
 self.percent_silenced.append(percent)
 self.f1_scores.append(f1_score)
 if self.save_csv:
 with open(self.csv_path, mode='a', newline='') as file:
 writer = csv.writer(file)
 writer.writerow([self.experiment_name, percent, f1_score])

 def plot(self, show=True, save_path=None):
 plt.plot(self.percent_silenced, self.f1_scores, marker='o')
 plt.title(f"F1-score - {self.experiment_name}")
 plt.xlabel("% Neuronas Silenciadas")
 plt.ylabel("F1-score")
 plt.ylim(0, 1)
 plt.grid(True)
 if save_path:
 plt.savefig(save_path, bbox_inches='tight')
 if show:
 plt.show()
 plt.clf()

In [ ]:
# === Profiling: wall-clock time + peak memory (added for R7.8 complexity analysis) ===
import time, sys, resource, csv, os
try:
 import torch as _torch
 _CUDA = _torch.cuda.is_available()
except Exception:
 _CUDA = False

class _Profiler:
 def __init__(self):
 self.rows = []
 self._stack = []
 def _rss_mb(self):
 # ru_maxrss: kilobytes on Linux, bytes on macOS
 m = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
 return m / 1024.0 if sys.platform.startswith("linux") else m / (1024.0 * 1024.0)
 def start(self, name):
 if _CUDA:
 _torch.cuda.synchronize(); _torch.cuda.reset_peak_memory_stats()
 self._stack.append((name, time.perf_counter()))
 def stop(self):
 name, t0 = self._stack.pop()
 if _CUDA:
 _torch.cuda.synchronize()
 dt = time.perf_counter() - t0
 gpu = (_torch.cuda.max_memory_allocated() / 1e6) if _CUDA else 0.0
 row = {"stage": name, "time_s": round(dt, 3),
 "peak_gpu_mb": round(gpu, 1), "proc_maxrss_mb": round(self._rss_mb(), 1)}
 self.rows.append(row)
 print(f"[profile] {name}: {dt:.2f}s | GPU peak {gpu:.0f} MB | proc RSS {row['proc_maxrss_mb']:.0f} MB")
 return row
 def save(self, path):
 d = os.path.dirname(path)
 if d:
 os.makedirs(d, exist_ok=True)
 with open(path, "w", newline="") as f:
 w = csv.DictWriter(f, fieldnames=["stage", "time_s", "peak_gpu_mb", "proc_maxrss_mb"])
 w.writeheader(); w.writerows(self.rows)
 print(f"[profile] saved {len(self.rows)} rows -> {path}")

PROFILE = _Profiler()
PROFILE_T0 = time.perf_counter()
print("[profile] profiler ready; CUDA available:", _CUDA)


# Configs

In [ ]:
import os
import torch
import json
import logging
import time
import numpy as np
import pandas as pd
import pickle
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import neurox.data.extraction.transformers_extractor as transformers_extractor
from neurox.data.writer import ActivationsWriter
import neurox.data.loader as data_loader
from transformers import AutoConfig
from tqdm import tqdm
import neurox.interpretation.linear_probe as linear_probe
import neurox.interpretation.utils as utils
import neurox.analysis.visualization as TransformersVisualizer
from sklearn.model_selection import train_test_split
from IPython.display import display
import neurox.interpretation.probeless as probeless
from neurox.interpretation.probeless import (
 get_neuron_ordering,
 get_neuron_ordering_for_all_tags
)
import ast
from torch.cuda.amp import autocast
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from matplotlib_venn import venn2
from neurox.interpretation.linear_probe import get_top_neurons
from sklearn.utils import shuffle

In [ ]:
import logging

# ==========================
# Configure Logging 
# ==========================

logger = logging.getLogger("synapse_logger")
logger.setLevel(logging.INFO)

# Avoid duplicates
if not logger.hasHandlers():

 # Handler 
 file_handler = logging.FileHandler("logs/synapse_extraction_csv_pth.log", mode="w")
 file_handler.setLevel(logging.INFO)

 # Handler 
 console_handler = logging.StreamHandler()
 console_handler.setLevel(logging.INFO)

 # Format
 formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
 file_handler.setFormatter(formatter)
 console_handler.setFormatter(formatter)

 # Add handlers to main logger
 logger.addHandler(file_handler)
 logger.addHandler(console_handler)

logger.info("Logging configured")


In [ ]:
# ==========================
# SYNAPSE Model Configuration
# ==========================

# Select the model (options: "BERT", "BigBird", "DistilBERT", "Longformer")
MODEL = "BigBird"

# Paths based on model name
BASE_PATH = f"data/{MODEL}"
input_csv = f"{BASE_PATH}/{MODEL}_tokens_PT.csv"
output_csv = f"{BASE_PATH}/reduced/{MODEL}_tokens_reduced.csv"
labels_output_path = f"{BASE_PATH}/labels_numeric.txt"
label_mapping_path = f"{BASE_PATH}/labels_mapping.json"
activations_file = f"{BASE_PATH}/activations.json"
weights_path = f"{BASE_PATH}/best_model_{MODEL}.pth"

# Number of labels
NUM_LABELS = 5


# HuggingFace model mapping
MODEL_HF = {
 "BERT": "bert-base-uncased",
 "BigBird": "google/bigbird-roberta-base",
 "DistilBERT": "distilbert-base-uncased",
 "Longformer": "allenai/longformer-base-4096"
}[MODEL]

# Device selection
device = torch.device("cuda"if torch.cuda.is_available() else "cpu")


# --- Global label encoding (MUST match the model's class space) ---
# The malware CSV labels are folder paths; the model was trained with LabelEncoder
# (alphabetical) on them, so we fit ONE encoder on the full file and use it EVERYWHERE
# (probe + eval). NORMAL_IDX is derived (not hardcoded).
from sklearn.preprocessing import LabelEncoder as _LE
_lab_full = pd.read_csv(input_csv, usecols=["label"])["label"].astype(str)
LABEL_ENCODER = _LE().fit(_lab_full)
LABEL2IDX = {str(c): int(i) for i, c in enumerate(LABEL_ENCODER.classes_)}
NORMAL_IDX = next((i for c, i in LABEL2IDX.items() if "normal"in c.lower()), 2)
print(f"[labels] classes -> "+ str({os.path.basename(c): i for c, i in LABEL2IDX.items()}) + f"| NORMAL_IDX={NORMAL_IDX}")


In [ ]:
# ==========================
# Revision experiment controls (COST KNOBS) — tune before each run
# ==========================
# Everything new added for the revision runs for the CURRENT MODEL, so switching MODEL
# above and re-running the notebook reruns the full suite (existing attacks + new
# comparisons) for that model. These knobs bound the extra cost.

# Statistical seeds for the multi-seed protocol (R4.4 / R7.3). KEEP AT 1 for the first
# timing run; raise later once per-model wall-clock is known and the budget allows.
N_SEEDS = 5

# Random-neuron control (R1.3 / R4.2): number of random draws for the null distribution.
# 1 makes the p-value/std meaningless, so a few are needed; keep it modest for timing.
RANDOM_CONTROL_DRAWS = 20
SAMPLE_N = 200 # eval-sample size per seed (main time driver)

# Neuron percentages: SAME set as the existing global-silencing sweep (single source of
# truth for the whole pipeline — do NOT invent new ones). New silencing-based cells reuse these.
SWEEP_PCTS = [0.025, 0.05, 0.075, 0.10, 0.125, 0.15, 0.175, 0.20,
 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.65, 0.75, 0.8, 0.95]

# On/off switches for the NEW experiments (set False to skip and save time):
RUN_DETECTION_METRICS = True # R3.2 — cheap (post-processing of predictions)
RUN_RANDOM_CONTROL = True # R1.3/R4.2 — cost ~ RANDOM_CONTROL_DRAWS * len(SWEEP_PCTS) evals
RUN_MEANPOOL_ABLATION = True # R3.4/R7.6 — re-extracts a mean-pool activation set (extraction cost)
RUN_BITFLIP = True # R7.9 — attribution-guided exponent-MSB bit-flip (deterministic; cheap)
RUN_ATTRIBUTION = True # #8 R1.1/R3.3/R7.5 — probe vs conductance/act-grad (captum; HEAVY -> subset)
ATTRIBUTION_N_SAMPLES = 32 # examples for the attribution comparison (bounds conductance cost)
ATTRIBUTION_STEPS = 20 # captum conductance integration steps

print(f"[config] MODEL={MODEL} | N_SEEDS={N_SEEDS} | RANDOM_CONTROL_DRAWS={RANDOM_CONTROL_DRAWS} "
 f"| n_pcts={len(SWEEP_PCTS)} | detection={RUN_DETECTION_METRICS} "
 f"random_ctrl={RUN_RANDOM_CONTROL} meanpool={RUN_MEANPOOL_ABLATION} bitflip={RUN_BITFLIP} attribution={RUN_ATTRIBUTION}")


In [ ]:
# ==========================
# Load Model and Weights
# ==========================
from transformers import AutoConfig

model = AutoModelForSequenceClassification.from_pretrained(MODEL_HF, num_labels=NUM_LABELS)

# Load trained weights from disk
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print(f"Loaded {MODEL} with pretrained weights on {device}")

### Load dataset

In [ ]:
reduction_ratio = 1000 # absolute train_size for the probe (real run)

# ==========================
# Skip dataset reduction if already available
# ==========================
if os.path.exists(output_csv) and os.path.exists(labels_output_path):
 logger.info(f"Reduced dataset found: {output_csv}. Skipping reduction.")
 df_reduced = pd.read_csv(output_csv)
 with open(labels_output_path, "r") as f:
 labels = [int(line.strip()) for line in f] # Labels as integers
 with open(label_mapping_path, "r") as f:
 label_mapping = json.load(f) # Load label mapping
else:
 logger.info(f"Loading dataset from {input_csv}")

 chunk_size = 5000 
 total_rows = sum(1 for _ in open(input_csv)) - 1 # Total rows excluding header
 df_chunks = []

 logger.info(f"Processing {total_rows} rows in chunks of {chunk_size}...")

 with tqdm(total=total_rows, desc="Processing rows", unit="rows") as pbar:
 for chunk in pd.read_csv(input_csv, chunksize=chunk_size):
 # Convert `input_ids` from string to list of integers
 chunk['input_ids'] = chunk['input_ids'].apply(lambda x: list(map(int, x.strip("[]").split(","))))
 df_chunks.append(chunk)
 pbar.update(len(chunk))

 df = pd.concat(df_chunks, ignore_index=True)

 # ==========================
 # Encode labels as integers
 # ==========================
 df['label'] = LABEL_ENCODER.transform(df["label"].astype(str))
 label_mapping = LABEL2IDX

 # ==========================
 # Reduce dataset maintaining class proportions
 # ==========================
 df_reduced, _ = train_test_split(df, train_size=reduction_ratio, stratify=df["label"], random_state=42)
 labels = df_reduced["label"].tolist()

 # ==========================
 # Save reduced dataset and labels
 # ==========================
 df_reduced.to_csv(output_csv, index=False)
 with open(labels_output_path, "w") as f:
 for label in labels:
 f.write(str(label) + "\n")

 with open(label_mapping_path, "w") as f:
 json.dump(label_mapping, f, indent=4)

 logger.info(f"Reduced dataset saved to {output_csv}")
 logger.info(f"Numeric labels saved to {labels_output_path}")
 logger.info(f"Label mapping saved to {label_mapping_path}")


### Create Dataloader

In [ ]:

# Create DataLoader

class SyscallDataset(Dataset):
 def __init__(self, dataframe):
 self.data = dataframe

 def __len__(self):
 return len(self.data)

 def __getitem__(self, idx):
 input_ids = torch.tensor(self.data.iloc[idx]['input_ids'])
 label = torch.tensor(self.data.iloc[idx]['label'])
 return input_ids, label

# Initialize DataLoader with reduced dataset
dataset = SyscallDataset(df_reduced)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

logger.info("Dataloader created")

# Ensure `input_ids` are lists of integers
if isinstance(df_reduced["input_ids"].iloc[0], str):
 df_reduced["input_ids"] = df_reduced["input_ids"].apply(lambda x: list(map(int, x.strip("[]").split(","))))


# NeuroX

## Activation Extraction

In [ ]:
PROFILE.start("Malware: activation extraction")
if os.path.exists(activations_file):
 logger.info(f"Activations file found: {activations_file}. Skipping extraction.")
else:
 transformers_extractor.extract_representations(
 model, 
 df_reduced["input_ids"].tolist(), # Pass preprocessed tokens directly
 activations_file,
 device=device,
 )

 logger.info(f"Activations saved to {activations_file}")
PROFILE.stop()


## Load Activations

In [ ]:
activations, num_layers = data_loader.load_activations(activations_file)
logger.info(f"Loaded activations from {activations_file} with {num_layers} layers")

# Load sentence-level classification data using activations
tokens = data_loader.load_sentence_data(
 output_csv, labels_output_path, activations
)

# Create sentence-level tensors for classification
X, y, mapping = utils.create_tensors(
 tokens,
 activations,
 task_specific_tag="NN",
 task_type="classification"
)

label2idx, idx2label, src2idx, idx2src = mapping
logger.info("Created input/output tensors and label mappings for classification")

## Train linear probe

In [ ]:
PROFILE.start("Malware: probe training + ranking")
probe = linear_probe.train_logistic_regression_probe(X, y, lambda_l1=0.001, lambda_l2=0.001)
scores = linear_probe.evaluate_probe(probe, X, y, idx_to_class=idx2label)
logger.info(f"Probe evaluation results: {scores}")

top_neurons_probe, per_class_top_neurons = linear_probe.get_top_neurons(probe, percentage=0.1, class_to_idx=label2idx)
logger.info(f"Top global neurons: {top_neurons_probe}")
logger.info(f"Top neurons per class: {per_class_top_neurons}")
PROFILE.stop()


# Experiments

## Original performance

In [ ]:
df = pd.read_csv(input_csv)

# Select 50 random examples and reset index
sample_df = df.sample(n=50, random_state=42).reset_index(drop=True)

# Convert "input_ids"and "attention_mask"from string to list format
def parse_list(x):
 return ast.literal_eval(x)

sample_df['input_ids'] = sample_df['input_ids'].apply(parse_list)
sample_df['attention_mask'] = sample_df['attention_mask'].apply(parse_list)

# Encode labels to integers
label_encoder = LabelEncoder()
sample_df['label'] = LABEL_ENCODER.transform(sample_df['label'].astype(str))
labels_list = sample_df['label'].tolist()

predictions_list = []
model.eval()
torch.cuda.empty_cache()

for i in range(len(sample_df)):
 input_ids_tensor = torch.tensor(sample_df.loc[i, 'input_ids']).unsqueeze(0).to(device)
 attention_mask_tensor = torch.tensor(sample_df.loc[i, 'attention_mask']).unsqueeze(0).to(device)

 with torch.no_grad():
 with autocast():
 device = torch.device("cuda"if torch.cuda.is_available() else "cpu")
 outputs = model(input_ids=input_ids_tensor, attention_mask=attention_mask_tensor)
 logits = outputs['logits']
 pred = torch.argmax(logits, dim=1).item()
 predictions_list.append(pred)

 del input_ids_tensor, attention_mask_tensor, outputs, logits
 torch.cuda.empty_cache()

# Compute evaluation metrics
accuracy = accuracy_score(labels_list, predictions_list)
f1 = f1_score(labels_list, predictions_list, average='weighted')
report_dict = classification_report(labels_list, predictions_list, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()

# Round for readability
report_df = report_df.round(4)

# Add accuracy as a separate row
accuracy_row = pd.DataFrame({'precision': accuracy, 'recall': accuracy, 'f1-score': accuracy, 'support': sum(report_df['support'])}, index=['accuracy'])
report_df = pd.concat([report_df, accuracy_row])

# Save or append to CSV

experiment_title = "Sample of 30 - Full Model Evaluation"
csv_report_path = f"{BASE_PATH}/classification_report_sample_eval.csv"

# Remove incorrect 'accuracy' row if it exists
report_df = report_df.drop("accuracy", errors="ignore")

# Append correct accuracy row
accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
}, index=["overall_accuracy"])

# Combine
final_df = pd.concat([report_df, accuracy_row])

# Write to CSV with experiment title as a header
with open(csv_report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
final_df.to_csv(csv_report_path, mode="a")

logger.info(f"Appended classification report with title '{experiment_title}' to {csv_report_path}")


In [ ]:
logger = logging.getLogger(__name__)

def quick_baseline_f1(model, sample_df, labels_list):
 """
 Corre inferencia rápida y devuelve solo el F1-weighted para el modelo actual.
 Úsalo tras cada experimento para comprobar que el baseline no se contamina.
 """
 model.eval()
 preds = []
 for row in sample_df.itertuples():
 input_ids = torch.tensor(row.input_ids).unsqueeze(0).to(model.device)
 att_mask = torch.tensor(row.attention_mask).unsqueeze(0).to(model.device)
 with torch.no_grad():
 logits = model(input_ids=input_ids, attention_mask=att_mask).logits
 preds.append(int(logits.argmax(dim=-1)))
 f1w = f1_score(labels_list, preds, average="weighted", zero_division=0)
 logger.info(f"[Baseline Check] Weighted F1-score: {f1w:.4f}")
 return f1w

## Shortcut: reload model

In [ ]:
df = pd.read_csv(input_csv)

# Select 50 random examples and reset index
sample_df = df.sample(n=50, random_state=42).reset_index(drop=True)
# --- consistency fix: parse list columns (same as the Original-performance cell) ---
import ast
sample_df['input_ids'] = sample_df['input_ids'].apply(ast.literal_eval)
sample_df['attention_mask'] = sample_df['attention_mask'].apply(ast.literal_eval)
def load_model(model_hf: str,
 weights_path: str = None,
 num_labels: int = None,
 device: str = None):
 # dispositivo
 if device is None:
 device = "cuda"if torch.cuda.is_available() else "cpu"
 device = torch.device(device)

 # carga base
 kwargs = {}
 if num_labels is not None:
 kwargs["num_labels"] = num_labels
 model = AutoModelForSequenceClassification.from_pretrained(model_hf, **kwargs)

 # aplica checkpoint propio si se pasa
 if weights_path:
 state_dict = torch.load(weights_path, map_location=device)
 # strict=False por si tu state_dict no coincide exactamente (labels, etc.)
 model.load_state_dict(state_dict, strict=False)

 model.to(device)
 model.eval()
 return model


model = load_model(MODEL_HF, weights_path=weights_path, num_labels=NUM_LABELS, device=("cuda"if torch.cuda.is_available() else "cpu"))


## Silence and evaluate neurons. Function definition

### Full silencing

In [ ]:
def get_top_k_neurons_exact(probe, percentage: float) -> list[int]:
 """
 Return exactly N = round(total_neurons * percentage) neuron indices, sorted by importance.
 Importance is measured as the sum of absolute values of weights across all output classes.
 """
 weight_matrix = probe.linear.weight.detach().abs() # [num_classes, num_neurons]
 importance = weight_matrix.sum(dim=0).cpu().numpy() # [num_neurons]
 total_neurons = len(importance)
 top_n = round(total_neurons * percentage)

 sorted_indices = importance.argsort()[-top_n:] # Top-N by importance
 return sorted_indices.tolist()

In [ ]:
def make_cls_silence_hook(indices):
 indices = [int(i) for i in indices]
 indices_tensor = torch.tensor(indices, dtype=torch.long) if indices else None

 def hook(module, input, output):
 if output.dim() == 3:
 new_output = output.clone()
 cls_token = new_output[:, 0, :]
 mask = torch.ones_like(cls_token)
 if indices_tensor is not None:
 local_indices = indices_tensor.to(new_output.device)
 mask[:, local_indices] = 0.0
 new_output[:, 0, :] = cls_token * mask
 return new_output
 return output
 return hook

In [ ]:
def make_cls_silence_hook(indices):
 """
 Forward hook that zeros out the selected neuron indices in the CLS token.
 It is compatible with:
 - BERT / RoBERTa / BigBird / Longformer (tensor output)
 - DistilBERT (tuple output, usually (hidden_state,) or (hidden_state, attentions))
 """
 idxs = torch.tensor(indices, dtype=torch.long)

 def hook(module, inp, output):
 # 1) Unify output into a `hidden` tensor
 if isinstance(output, tuple):
 if len(output) == 0:
 # Nothing to do
 return output
 hidden = output[0]
 else:
 hidden = output

 # 2) Sanity checks
 if not hasattr(hidden, "dim") or hidden.dim() != 3:
 # Not a (batch, seq_len, hidden) tensor do nothing
 return output

 if idxs.numel() == 0:
 # No neuron to silence in this layer
 return output

 # 3) Clone and modify CLS token
 new_hidden = hidden.clone()
 cls_token = new_hidden[:, 0, :] # (batch, hidden_dim)
 cls_token[:, idxs] = 0.0 # silence selected neurons
 new_hidden[:, 0, :] = cls_token

 # 4) Rebuild structure depending on model
 if isinstance(output, tuple):
 # Keep any extra elements (e.g. attentions) untouched
 return (new_hidden,) + tuple(output[1:])
 else:
 return new_hidden

 return hook

In [ ]:
# ==========================
# Get encoder layers dynamically
# ==========================
def get_encoder_layers(model):
 if hasattr(model, "bert"):
 return model.bert.encoder.layer
 elif hasattr(model, "longformer"):
 return model.longformer.encoder.layer
 elif hasattr(model, "distilbert"):
 return model.distilbert.transformer.layer
 else:
 raise NotImplementedError("Unsupported model architecture.")

In [ ]:
def silence_top_global_percentage_and_evaluate(
 model,
 sample_df,
 labels_list,
 probe,
 label2idx,
 percentage=0.10,
 report_path=None,
 experiment_title=None
):
 hidden_dim = model.config.hidden_size
 num_layers = model.config.num_hidden_layers
 total_neurons = num_layers * hidden_dim

 top_neurons_global = get_top_k_neurons_exact(probe, percentage=percentage)
 logger.info(f"Silencing exactly {len(top_neurons_global)} neurons ({percentage:.2%} of total {total_neurons})")

 # Save neuron indices
 neurons_dir = f"{BASE_PATH}/neurons"
 os.makedirs(neurons_dir, exist_ok=True)
 json_path = f"{neurons_dir}/top_{int(percentage * 100)}p_neurons_global.json"
 with open(json_path, "w") as f:
 json.dump(top_neurons_global, f, indent=4)
 logger.info(f"Saved neuron indices to {json_path}")

 # Register hooks per layer (compatible with BERT, DistilBERT, etc.)
 encoder_layers = get_encoder_layers(model)
 hook_handles = []
 for i in range(num_layers):
 indices_layer = [idx - i * hidden_dim for idx in top_neurons_global if i * hidden_dim <= idx < (i + 1) * hidden_dim]
 if indices_layer:
 logger.info(f"Layer {i}: silencing {len(indices_layer)} neurons")
 # Use 'output' submodule if exists (BERT, RoBERTa), else register on main layer (DistilBERT)
 if hasattr(encoder_layers[i], "output"):
 handle = encoder_layers[i].output.register_forward_hook(make_cls_silence_hook(indices_layer))
 else:
 handle = encoder_layers[i].register_forward_hook(make_cls_silence_hook(indices_layer))
 hook_handles.append(handle)

 # Inference
 model.eval()
 predictions = []
 for i in range(len(sample_df)):
 input_ids_tensor = torch.tensor(sample_df.loc[i, 'input_ids']).unsqueeze(0).to(model.device)
 attention_mask_tensor = torch.tensor(sample_df.loc[i, 'attention_mask']).unsqueeze(0).to(model.device)

 with torch.no_grad():
 outputs = model(input_ids=input_ids_tensor, attention_mask=attention_mask_tensor)
 logits = outputs['logits']
 pred = torch.argmax(logits, dim=1).item()
 predictions.append(pred)

 del input_ids_tensor, attention_mask_tensor, outputs, logits
 torch.cuda.empty_cache()

 # Metrics
 accuracy = accuracy_score(labels_list, predictions)
 f1 = f1_score(labels_list, predictions, average='weighted')
 report_dict = classification_report(labels_list, predictions, output_dict=True)
 report_df = pd.DataFrame(report_dict).transpose().round(4)
 report_df = report_df.drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 # Save classification report
 if report_path is None:
 report_path = f"{BASE_PATH}/results/full_silencing.csv"
 os.makedirs(os.path.dirname(report_path), exist_ok=True)

 if experiment_title is None:
 experiment_title = f"Silencing top {percentage:.2%} global neurons"

 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Accuracy after silencing: {accuracy:.4f}")
 logger.info(f"Weighted F1 Score: {f1:.4f}")
 logger.info(f"Classification report saved to {report_path}")

 # Remove all hooks
 for handle in hook_handles:
 handle.remove()
 logger.info("All hooks removed after evaluation")

## Global impact

In [ ]:
percentages = SWEEP_PCTS # unified: single source of truth (control panel)

for pct in percentages:
 silence_top_global_percentage_and_evaluate(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 probe=probe,
 label2idx=label2idx,
 percentage=pct,
 experiment_title=f"Silencing {pct*100:.1f}% Global Neurons"
 )


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO
from pathlib import Path

def plot_attack_report(
 report_path: str,
 classes: list = None,
 label_map: dict = None,
 global_metric: str = "weighted", # "weighted"or "macro"
 title: str = None,
 out_path: str = None
):
 """
 Plot a single-figure summary for an attack report file with a muted, readable style.
 - One line per class (muted colors, thin lines).
 - One dashed black global line (macro/weighted).
 """
 if not os.path.exists(report_path):
 print(f"[plot] skipped (missing file): {report_path}")
 return
 text = Path(report_path).read_text(encoding="utf-8", errors="ignore")
 lines = text.splitlines()

 # --- Split into sections by '# ' headers
 sections = []
 i = 0
 while i < len(lines):
 if lines[i].startswith("# "):
 header = lines[i][2:].strip()
 j = i + 1
 block_lines = []
 while j < len(lines) and not lines[j].startswith("# "):
 block_lines.append(lines[j])
 j += 1
 sections.append((header, "\n".join(block_lines).strip()))
 i = j
 else:
 i += 1
 if not sections:
 raise ValueError(f"No sections found in {report_path}.")

 # --- Parse each section
 percent_re = re.compile(r"([0-9]*\.?[0-9]+)\s*%")
 recs_pc, recs_g = [], []
 for header, block in sections:
 if not block:
 continue
 m = percent_re.search(header)
 if not m:
 continue
 percent = float(m.group(1)) / 100.0

 try:
 df = pd.read_csv(StringIO(block))
 except Exception:
 df = pd.read_csv(StringIO(block), sep=';')
 if df.empty:
 continue

 f1_col = 'f1-score' if 'f1-score' in df.columns else ('f1' if 'f1' in df.columns else None)
 if f1_col is None:
 raise ValueError("Missing 'f1-score' column in a section.")
 label_col = df.columns[0]
 df[label_col] = df[label_col].astype(str)

 macro_row = df[df[label_col].str.lower().str.strip() == 'macro avg']
 weighted_row = df[df[label_col].str.lower().str.strip() == 'weighted avg']
 macro = float(macro_row[f1_col].iloc[0]) if not macro_row.empty else np.nan
 weighted = float(weighted_row[f1_col].iloc[0]) if not weighted_row.empty else np.nan
 recs_g.append({"percent": percent, "macro": macro, "weighted": weighted})

 for _, r in df.iterrows():
 raw_lbl = str(r[label_col]).strip()
 if raw_lbl.lower() in ('macro avg','weighted avg','overall_accuracy'):
 continue
 try:
 f1v = float(r[f1_col])
 except Exception:
 continue
 display_lbl = label_map.get(raw_lbl, raw_lbl) if isinstance(label_map, dict) else raw_lbl
 recs_pc.append({"percent": percent, "class": display_lbl, "f1": f1v})

 if not recs_pc or not recs_g:
 raise ValueError("No valid per-class or global data extracted.")

 df_pc = pd.DataFrame(recs_pc)
 df_g = pd.DataFrame(recs_g)
 pc_mean = df_pc.groupby(['class','percent'])['f1'].mean().reset_index()
 g_mean = df_g.groupby('percent')[['macro','weighted']].mean().reset_index()

 # ---- Style: muted palette & thinner lines
 plt.rcParams.update({
 "figure.figsize": (9.5, 5.0), # more breathing room
 "axes.grid": True,
 "grid.linestyle": "--",
 "grid.alpha": 0.18, # very subtle grid
 "axes.labelsize": 10.5,
 "axes.titlesize": 14,
 "xtick.labelsize": 9.5,
 "ytick.labelsize": 9.5,
 "legend.fontsize": 9.5,
 "lines.linewidth": 1.5, # thinner class lines
 "lines.markersize": 4, # smaller markers
 })
 # Muted, colorblind-friendly palette (desaturated)
 palette = [
 "#5B84B1", # muted blue
 "#7AA974", # muted green
 "#C26D6D", # muted red
 "#C9B458", # muted mustard
 "#9A86B4", # muted purple
 "#7FB0A6", # muted teal
 "#A38F85", # muted taupe (fallback)
 "#B7B7B7", # muted gray (fallback)
 ]
 global_style = {"color": "#222222", "linestyle": (0, (6, 4)), "linewidth": 2.0}

 # ---- Determine class order
 if classes is None:
 classes_order = list(pc_mean['class'].unique())
 else:
 classes_order = [label_map.get(c, c) if isinstance(label_map, dict) else c for c in classes]

 fig, ax = plt.subplots()

 # Per-class lines (slightly transparent so they don't overpower)
 for i, cls in enumerate(classes_order):
 sub = pc_mean[pc_mean['class'] == cls].sort_values('percent')
 if sub.empty:
 continue
 ax.plot(sub['percent'].values, sub['f1'].values, marker="o",
 label=str(cls), alpha=0.9, color=palette[i % len(palette)])

 # Global metric (weighted or macro)
 gm = 'macro' if str(global_metric).lower().startswith('m') else 'weighted'
 gsub = g_mean.sort_values('percent')
 ax.plot(gsub['percent'].values, gsub[gm].values, marker="o",
 label=f"F1-{gm}", **global_style)

 ax.set_xlabel("Silenced percentage")
 ax.set_ylabel("F1-score")
 ax.set_ylim(0.0, 1.0) # keeps vertical scale consistent
 ax.set_title(title or Path(report_path).stem)
 ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, title="Classes")
 plt.tight_layout()
 if out_path:
 Path(out_path).parent.mkdir(parents=True, exist_ok=True)
 plt.savefig(out_path, bbox_inches="tight", dpi=300)
 plt.close()
 else:
 plt.show()

In [ ]:
report = f"{BASE_PATH}/results/full_silencing.csv"
plot_attack_report(report_path=report,
 classes=['0','1','2','3','4'],
 global_metric='weighted',
 title=f"Global Silencing ({MODEL})",
 out_path=f"{BASE_PATH}/figs/{MODEL}_silencing.png")

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO
from pathlib import Path
CLASSES = ['0', '1', '2', '3', '4']
LABEL_MAP = {
 '0': 'Bdvl',
 '1': 'RansomwarePoC',
 '2': 'TheTick',
 '3': 'Normal',
 '4': 'Bashlite'
}

RESULTS_DIR = Path(f"{BASE_PATH}/results")
FIGS_DIR = Path(f"{BASE_PATH}/figs")
FIGS_DIR.mkdir(exist_ok=True, parents=True)

model_suffix = MODEL.lower() # "BigBird"-> "bigbird"

def plot_attack_report(
 report_path: str,
 classes: list,
 label_map: dict,
 global_metric: str = "weighted",
 title: str = None,
 out_path: str = None
):
 """Plot global neuron silencing with class names (not numeric labels)."""

 # ---------- read ----------
 if not os.path.exists(report_path):
 print(f"[plot] skipped (missing file): {report_path}")
 return
 text = Path(report_path).read_text(encoding="utf-8", errors="ignore")
 lines = text.splitlines()

 sections, i = [], 0
 while i < len(lines):
 if lines[i].startswith("# "):
 header = lines[i][2:].strip()
 j = i + 1
 block = []
 while j < len(lines) and not lines[j].startswith("# "):
 block.append(lines[j]); j += 1
 sections.append((header, "\n".join(block).strip()))
 i = j
 else:
 i += 1

 percent_re = re.compile(r"([0-9]*\.?[0-9]+)\s*%")
 recs_pc, recs_g = [], []

 # ---------- parse ----------
 for header, block in sections:
 if not block:
 continue

 m = percent_re.search(header)
 if not m:
 continue

 percent = float(m.group(1)) / 100.0

 try:
 df = pd.read_csv(StringIO(block))
 except Exception:
 df = pd.read_csv(StringIO(block), sep=";")

 f1_col = "f1-score"if "f1-score"in df.columns else "f1"
 label_col = df.columns[0]

 df[label_col] = df[label_col].astype(str)

 macro_row = df[df[label_col].str.lower().str.strip() == "macro avg"]
 weighted_row = df[df[label_col].str.lower().str.strip() == "weighted avg"]

 macro = float(macro_row[f1_col].iloc[0]) if not macro_row.empty else np.nan
 weighted = float(weighted_row[f1_col].iloc[0]) if not weighted_row.empty else np.nan

 recs_g.append({"percent": percent, "macro": macro, "weighted": weighted})

 # per-class metrics
 for _, r in df.iterrows():
 raw_label = str(r[label_col]).strip()
 if raw_label.lower() in ("macro avg", "weighted avg", "overall_accuracy"):
 continue
 try:
 f1v = float(r[f1_col])
 except:
 continue

 display = label_map.get(raw_label, raw_label)
 recs_pc.append({"percent": percent, "class": display, "f1": f1v})

 df_pc = pd.DataFrame(recs_pc)
 df_g = pd.DataFrame(recs_g)

 pc_mean = df_pc.groupby(["class", "percent"])["f1"].mean().reset_index()
 g_mean = df_g.groupby("percent")[["macro", "weighted"]].mean().reset_index()

 # ---------- STYLE ----------
 plt.rcParams.update({
 "figure.figsize": (10, 6.2),
 "axes.grid": True,
 "grid.linestyle": "--",
 "grid.alpha": 0.15,
 "axes.labelsize": 18,
 "axes.titlesize": 20,
 "xtick.labelsize": 17,
 "ytick.labelsize": 17,
 "legend.fontsize": 19,
 "legend.title_fontsize": 19,
 "lines.linewidth": 1.8,
 "lines.markersize": 7,
 })

 blue_palette = ["#D6E2F4", "#C1D4EF", "#9FBBE2", "#7EA3D6", "#5E8BC6", "#4976B2", "#3A6296"]

 def mono_palette(n):
 idx = np.linspace(0, len(blue_palette)-1, n).round().astype(int)
 return [blue_palette[i] for i in idx]

 shades = mono_palette(len(classes))
 markers = ['s','^','D','v','P']

 global_style = {
 "color": "#111111",
 "linestyle": (0, (6,4)),
 "linewidth": 2.4,
 "marker": "o",
 "markersize": 7
 }

 # ---------- PLOT ----------
 fig, ax = plt.subplots()
 ax.set_ylim(0, 1.0)
 ax.set_yticks(np.linspace(0, 1, 6))

 # per-class curves
 for i, raw_cls in enumerate(classes):
 disp = label_map.get(raw_cls, raw_cls)
 sub = pc_mean[pc_mean["class"] == disp].sort_values("percent")

 ax.plot(
 sub["percent"],
 sub["f1"],
 marker=markers[i % len(markers)],
 color=shades[i],
 label=disp,
 alpha=0.95
 )

 # global line
 gm = "macro"if global_metric.lower().startswith("m") else "weighted"
 ax.plot(g_mean["percent"], g_mean[gm], label=f"F1-{gm}", **global_style)

 ax.set_xlabel("Silenced percentage")
 ax.set_ylabel("F1-score")
 ax.set_title(title or Path(report_path).stem)

 ax.set_title(title or auto_title, fontsize=24, pad=100)

 # ====== ORDENAR LEYENDA ======
 handles, labels = ax.get_legend_handles_labels()
 order = sorted(range(len(labels)), key=lambda i: (0 if "(target)"in labels[i] else 1, labels[i]))
 handles = [handles[i] for i in order]
 labels = [labels[i] for i in order]

 # ====== CREAR LEYENDA DEBAJO DEL TÍTULO (SIN SOLAPAR) ======
 legend = fig.legend(
 handles,
 labels,
 loc="upper center",
 bbox_to_anchor=(0.5, 1.02), # justo debajo del título
 ncol=3,
 frameon=False,
 title="Classes",
 fontsize=18,
 title_fontsize=20
 )

 # ====== RESERVAR ESPACIO ARRIBA PARA TÍTULO + LEYENDA ======
 fig.subplots_adjust(top=0.78) 

 if out_path:
 Path(out_path).parent.mkdir(exist_ok=True, parents=True)
 plt.savefig(out_path, dpi=300, bbox_inches="tight")
 plt.close()
 else:
 plt.show()

In [ ]:
plot_attack_report(
 report_path=f"{BASE_PATH}/results/bigbird_global_silencing.csv",
 classes=['0','1','2','3','4'],
 label_map=LABEL_MAP,
 global_metric="weighted",
 title=f"Global Silencing ({MODEL})",
 out_path=f"{BASE_PATH}/figs/{MODEL}_silencing_blue.png"
)

## Impact per class

In [ ]:
def get_top_k_neurons_for_class_exact(probe, percentage: float, class_to_idx: dict, class_id: int) -> list[int]:
 """
 Return top-k neurons most important for a specific class, measured by absolute weight.
 """
 weight_matrix = probe.linear.weight.detach().abs() # [num_classes, num_neurons]
 class_weights = weight_matrix[class_id] # [num_neurons]
 total_neurons = class_weights.size(0)
 top_n = round(percentage * total_neurons)
 top_indices = class_weights.cpu().numpy().argsort()[-top_n:]
 return top_indices.tolist()

In [ ]:

def make_cls_silence_hook_class(indices):
 idxs = torch.tensor(indices, dtype=torch.long)

 def hook(module, input, output):
 # DistilBERT / some HF configs: output is tuple take first element
 if isinstance(output, tuple):
 main = output[0]
 else:
 main = output

 # If not a tensor (rare), skip
 if not torch.is_tensor(main):
 return output

 # CLS is at position 0
 if main.dim() == 3 and idxs.numel() > 0:
 new_main = main.clone()
 cls = new_main[:, 0, :]
 cls[:, idxs] = 0
 new_main[:, 0, :] = cls

 # Rebuild structure if needed
 if isinstance(output, tuple):
 return (new_main,) + output[1:]
 else:
 return new_main

 return output

 return hook

In [ ]:
'''
# ==========================
# Get encoder layers dynamically
# ==========================
def get_encoder_layers(model):
 if hasattr(model, "bert"):
 return model.bert.encoder.layer
 elif hasattr(model, "longformer"):
 return model.longformer.encoder.layer
 elif hasattr(model, "distilbert"):
 return model.distilbert.transformer.layer
 else:
 raise NotImplementedError("Unsupported model architecture.")

# ==========================
# Silence and evaluate top per-class neurons
# ==========================

def silence_top_class_percentage_and_evaluate(
 model,
 sample_df,
 labels_list,
 probe,
 label2idx,
 class_id: int,
 percentage: float = 0.1,
 report_path: str = None,
 experiment_title: str = None
):
 class_name = f"class_{class_id}"
 hidden_dim = model.config.hidden_size
 num_layers = model.config.num_hidden_layers
 total_neurons = num_layers * hidden_dim

 # Get top neurons for specific class
 top_class_neurons = get_top_k_neurons_for_class_exact(
 probe, percentage=percentage, class_to_idx=label2idx, class_id=class_id
 )

 logger.info(f"Silencing {len(top_class_neurons)} neurons for class {class_id} ({percentage:.2%} of total)")

 # Save neurons to JSON
 neurons_dir = f"{BASE_PATH}/neurons"
 os.makedirs(neurons_dir, exist_ok=True)
 json_path = f"{neurons_dir}/top_{int(percentage * 100)}p_neurons_{class_name}.json"
 with open(json_path, "w") as f:
 json.dump(top_class_neurons, f, indent=4)
 logger.info(f"Saved neuron indices to {json_path}")

 # Register hooks per layer
 encoder_layers = get_encoder_layers(model)
 hook_handles = []
 for i in range(num_layers):
 indices_layer = [idx - i * hidden_dim for idx in top_class_neurons if i * hidden_dim <= idx < (i + 1) * hidden_dim]
 if indices_layer:
 logger.info(f"Layer {i}: silencing {len(indices_layer)} neurons for class {class_id}")
 handle = encoder_layers[i].output.register_forward_hook(make_cls_silence_hook(indices_layer))
 hook_handles.append(handle)

 # Inference
 model.eval()
 predictions = []
 for i in range(len(sample_df)):
 input_ids_tensor = torch.tensor(sample_df.loc[i, 'input_ids']).unsqueeze(0).to(model.device)
 attention_mask_tensor = torch.tensor(sample_df.loc[i, 'attention_mask']).unsqueeze(0).to(model.device)

 with torch.no_grad():
 outputs = model(input_ids=input_ids_tensor, attention_mask=attention_mask_tensor)
 logits = outputs['logits']
 pred = torch.argmax(logits, dim=1).item()
 predictions.append(pred)

 del input_ids_tensor, attention_mask_tensor, outputs, logits
 torch.cuda.empty_cache()

 # Metrics
 accuracy = accuracy_score(labels_list, predictions)
 f1 = f1_score(labels_list, predictions, average='weighted')
 report_dict = classification_report(labels_list, predictions, output_dict=True)
 report_df = pd.DataFrame(report_dict).transpose().round(4)
 report_df = report_df.drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 # Save classification report
 if report_path is None:
 report_path = f"{BASE_PATH}/results/class_silencing{class_id}.csv"
 os.makedirs(os.path.dirname(report_path), exist_ok=True)

 if experiment_title is None:
 experiment_title = f"Silencing top {percentage:.2%} neurons for class {class_id}"

 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Accuracy after class-specific silencing: {accuracy:.4f}")
 logger.info(f"Weighted F1 Score: {f1:.4f}")
 logger.info(f"Classification report saved to {report_path}")

 # Remove all hooks
 for handle in hook_handles:
 handle.remove()
 logger.info("All hooks removed after evaluation")
'''

In [ ]:

# ==========================
# Get encoder layers dynamically
# ==========================
def get_encoder_layers(model):
 if hasattr(model, "bert"):
 return model.bert.encoder.layer
 elif hasattr(model, "longformer"):
 return model.longformer.encoder.layer
 elif hasattr(model, "distilbert"):
 return model.distilbert.transformer.layer
 else:
 raise NotImplementedError("Unsupported model architecture.")

def silence_top_class_percentage_and_evaluate(
 model,
 sample_df,
 labels_list,
 probe,
 label2idx,
 class_id: int,
 percentage: float = 0.1,
 report_path: str = None,
 experiment_title: str = None
):
 class_name = f"class_{class_id}"
 hidden_dim = model.config.hidden_size
 num_layers = model.config.num_hidden_layers
 total_neurons = num_layers * hidden_dim

 # Get top neurons for the specific class
 top_class_neurons = get_top_k_neurons_for_class_exact(
 probe, percentage=percentage, class_to_idx=label2idx, class_id=class_id
 )
 logger.info(
 f"Silencing {len(top_class_neurons)} neurons for class {class_id} "
 f"({percentage:.2%} of total {total_neurons})"
 )

 # Save neuron indices to JSON
 neurons_dir = f"{BASE_PATH}/neurons"
 os.makedirs(neurons_dir, exist_ok=True)
 json_path = f"{neurons_dir}/top_{int(percentage * 100)}p_neurons_{class_name}.json"
 with open(json_path, "w") as f:
 json.dump(top_class_neurons, f, indent=4)
 logger.info(f"Saved neuron indices to {json_path}")

 # Register hooks per layer
 encoder_layers = get_encoder_layers(model)
 hook_handles = []

 for i in range(num_layers):
 # Select neuron indices belonging to this layer in the flattened indexing
 indices_layer = [
 idx - i * hidden_dim
 for idx in top_class_neurons
 if i * hidden_dim <= idx < (i + 1) * hidden_dim
 ]

 if not indices_layer:
 continue

 logger.info(
 f"Layer {i}: silencing {len(indices_layer)} neurons for class {class_id}"
 )

 layer = encoder_layers[i]

 # 1) BERT / RoBERTa / BigBird / Longformer they have `.output`
 # 2) DistilBERT no `.output`, hook the whole block instead
 if hasattr(layer, "output"):
 handle = layer.output.register_forward_hook(
 make_cls_silence_hook_class(indices_layer)
 )
 else:
 handle = layer.register_forward_hook(
 make_cls_silence_hook_class(indices_layer)
 )

 hook_handles.append(handle)

 # Inference under class-specific silencing
 model.eval()
 predictions = []
 for i in range(len(sample_df)):
 input_ids_tensor = torch.tensor(sample_df.loc[i, 'input_ids']).unsqueeze(0).to(model.device)
 attention_mask_tensor = torch.tensor(sample_df.loc[i, 'attention_mask']).unsqueeze(0).to(model.device)

 with torch.no_grad():
 outputs = model(input_ids=input_ids_tensor, attention_mask=attention_mask_tensor)
 logits = outputs['logits']
 pred = torch.argmax(logits, dim=1).item()
 predictions.append(pred)

 del input_ids_tensor, attention_mask_tensor, outputs, logits
 torch.cuda.empty_cache()

 # Metrics
 accuracy = accuracy_score(labels_list, predictions)
 f1 = f1_score(labels_list, predictions, average='weighted')
 report_dict = classification_report(labels_list, predictions, output_dict=True)
 report_df = pd.DataFrame(report_dict).transpose().round(4)
 report_df = report_df.drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 # Save classification report
 if report_path is None:
 report_path = f"{BASE_PATH}/results/class_silencing{class_id}.csv"
 os.makedirs(os.path.dirname(report_path), exist_ok=True)

 if experiment_title is None:
 experiment_title = f"Silencing top {percentage:.2%} neurons for class {class_id}"

 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Accuracy after class-specific silencing: {accuracy:.4f}")
 logger.info(f"Weighted F1 Score: {f1:.4f}")
 logger.info(f"Classification report saved to {report_path}")

 # Remove all hooks
 for handle in hook_handles:
 handle.remove()
 logger.info("All hooks removed after evaluation")

### Plotting

In [ ]:
def plot_class_silencing_report(
 report_path: str,
 target_class: str,
 classes: list,
 label_map: dict = None,
 theme: str = None,
 global_metric: str = "weighted",
 title: str = None,
 out_path: str = None,
 save: bool = True,
 show: bool = False,
 **_ignore
):
 import re
 import numpy as np
 import pandas as pd
 import matplotlib.pyplot as plt
 from io import StringIO
 from pathlib import Path

 target_class = str(target_class)
 if label_map is None:
 label_map = {str(c): str(c) for c in classes}

 if not os.path.exists(report_path):
 print(f"[plot] skipped (missing file): {report_path}")
 return
 text = Path(report_path).read_text(encoding="utf-8", errors="ignore")
 lines = text.splitlines()

 sections = []
 i = 0
 while i < len(lines):
 if lines[i].startswith("# "):
 header = lines[i][2:].strip()
 j = i + 1
 block = []
 while j < len(lines) and not lines[j].startswith("# "):
 block.append(lines[j]); j += 1
 sections.append((header, "\n".join(block)))
 i = j
 else:
 i += 1

 percent_re = re.compile(r"([0-9]*\.?[0-9]+)\s*%")
 rec_pc, rec_g = [], []

 for header, block in sections:
 m = percent_re.search(header)
 if not m or not block.strip():
 continue

 pct = float(m.group(1)) / 100.0

 try:
 df = pd.read_csv(StringIO(block))
 except Exception:
 df = pd.read_csv(StringIO(block), sep=";")

 f1_col = "f1-score"if "f1-score"in df.columns else "f1"
 label_col = df.columns[0]
 df[label_col] = df[label_col].astype(str)

 macro = df[df[label_col].str.lower().str.strip() == "macro avg"][f1_col]
 weighted = df[df[label_col].str.lower().str.strip() == "weighted avg"][f1_col]

 rec_g.append({
 "percent": pct,
 "macro": float(macro.iloc[0]) if len(macro) > 0 else np.nan,
 "weighted": float(weighted.iloc[0]) if len(weighted) > 0 else np.nan
 })

 for _, r in df.iterrows():
 raw = r[label_col].strip()
 if raw.lower() in ("macro avg", "weighted avg", "overall_accuracy"):
 continue
 rec_pc.append({
 "percent": pct,
 "raw": raw,
 "class": label_map.get(raw, raw),
 "f1": float(r[f1_col])
 })

 df_pc = pd.DataFrame(rec_pc)
 df_g = pd.DataFrame(rec_g)

 pc = df_pc.groupby(["raw", "class", "percent"])["f1"].mean().reset_index()
 gl = df_g.groupby("percent")[["macro", "weighted"]].mean().reset_index()

 # ---------- STYLE: IGUAL QUE GLOBAL ----------
 plt.rcParams.update({
 "figure.figsize": (10, 6.2),
 "axes.grid": True,
 "grid.linestyle": "--",
 "grid.alpha": 0.15,
 "axes.labelsize": 18,
 "axes.titlesize": 20,
 "xtick.labelsize": 17,
 "ytick.labelsize": 17,
 "legend.fontsize": 19,
 "legend.title_fontsize": 19,
 "lines.linewidth": 1.8,
 "lines.markersize": 7,
 })

 blue_palette = ["#D6E2F4","#C1D4EF","#9FBBE2","#7EA3D6","#5E8BC6","#4976B2","#3A6296"]
 def mono_palette(n):
 idx = np.linspace(0, len(blue_palette)-1, max(1, n)).round().astype(int).tolist()
 return [blue_palette[i] for i in idx]

 classes_order = classes
 shades = mono_palette(len(classes_order))
 markers = ['s','^','D','v','P','X','<','>','h','*']

 target_color = "#4976B2"# azul más oscuro
 grey_line = "#C9C9C9"
 grey_edge = "#8E8E8D"

 global_style = {
 "color": "#111111",
 "linestyle": (0, (6, 4)),
 "linewidth": 2.4,
 "marker": "o",
 "markersize": 7
 }

 fig, ax = plt.subplots()

 ax.set_ylim(0, 1.0)
 ax.set_yticks(np.linspace(0, 1, 6))

 # Todas las clases, resaltando la target
 for i, raw_c in enumerate(classes_order):
 sub = pc[pc["raw"] == raw_c].sort_values("percent")
 disp = label_map.get(raw_c, raw_c)
 marker = markers[i % len(markers)]

 if str(raw_c) == target_class:
 ax.plot(sub["percent"], sub["f1"],
 marker=marker,
 color=target_color,
 linewidth=2.4,
 markersize=7,
 label=f"{disp} (target)")
 else:
 ax.plot(sub["percent"], sub["f1"],
 marker=marker,
 color=grey_line,
 markeredgecolor=grey_edge,
 linewidth=1.2,
 markersize=6,
 alpha=0.75,
 label=disp)

 gm = "macro"if global_metric.startswith("m") else "weighted"
 ax.plot(gl["percent"], gl[gm], label=f"F1-{gm}", **global_style)

 class_name = label_map.get(target_class, target_class)
 p = Path(report_path)
 auto_title = f"Class-targeted Silencing (target={class_name})"
 ax.set_title(title or auto_title)

 ax.set_xlabel("Silenced percentage")
 ax.set_ylabel("F1-score")

 # ====== TÍTULO ARRIBA ======
 ax.set_title(title or auto_title, fontsize=24, pad=100)

 # ====== ORDENAR LEYENDA ======
 handles, labels = ax.get_legend_handles_labels()
 order = sorted(range(len(labels)), key=lambda i: (0 if "(target)"in labels[i] else 1, labels[i]))
 handles = [handles[i] for i in order]
 labels = [labels[i] for i in order]

 # ====== CREAR LEYENDA DEBAJO DEL TÍTULO (SIN SOLAPAR) ======
 legend = fig.legend(
 handles,
 labels,
 loc="upper center",
 bbox_to_anchor=(0.5, 1.02), # justo debajo del título
 ncol=3,
 frameon=False,
 title="Classes",
 fontsize=18,
 title_fontsize=20
 )

 # ====== RESERVAR ESPACIO ARRIBA PARA TÍTULO + LEYENDA ======
 fig.subplots_adjust(top=0.78) 

 if save:
 if out_path is None:
 out_path = p.with_name(f"class_silencing_target_{class_name}.png")
 plt.savefig(out_path, dpi=300, bbox_inches="tight")

 if show:
 plt.show()

 plt.close()

In [ ]:
from pathlib import Path

CLASSES = ['0', '1', '2', '3', '4']
LABEL_MAP = {
 '0': 'Bdvl',
 '1': 'RansomwarePoC',
 '2': 'TheTick',
 '3': 'Normal',
 '4': 'Bashlite'
}

RESULTS_DIR = Path(f"{BASE_PATH}/results")
FIGS_DIR = Path(f"{BASE_PATH}/figs")
FIGS_DIR.mkdir(exist_ok=True, parents=True)

model_suffix = MODEL.lower() # "BigBird"-> "bigbird"

for cls in CLASSES:
 report_file = RESULTS_DIR / f"class_silencing{cls}{model_suffix}.csv"

 if not report_file.exists():
 print(f"Report not found for class {cls}: {report_file}")
 continue

 output_png = FIGS_DIR / f"{MODEL}_class{cls}_silencing.png"

 print(f"Generating plot for target class {cls} {LABEL_MAP[cls]}")
 plot_class_silencing_report(
 report_path=str(report_file),
 target_class=cls,
 classes=CLASSES,
 label_map=LABEL_MAP,
 global_metric="weighted",
 out_path=str(output_png),
 save=True,
 show=False
 )

print("Done. Plots saved in:", FIGS_DIR)

In [ ]:
percentages = SWEEP_PCTS # unified: single source of truth (control panel)
target_class_id = 4

for pct in percentages:
 silence_top_class_percentage_and_evaluate(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 probe=probe,
 label2idx=label2idx,
 class_id=target_class_id,
 percentage=pct,
 experiment_title=f"Silencing {pct*100:.1f}% of Neurons for Class {target_class_id}"
 )
report = f"{BASE_PATH}/results/class_silencing4.csv"
plot_class_silencing_report(
 report_path=report,
 target_class='4',
 classes=['0','1','2','3','4'],
 global_metric='weighted',
 theme='blue',
 title=f'Class-targeted Silencing (target=4, {MODEL})',
 out_path=f"{BASE_PATH}/figs/silencing_target4_{MODEL}.png"
)

In [ ]:
percentages = SWEEP_PCTS # unified: single source of truth (control panel)

target_class_id = 3

for pct in percentages:
 silence_top_class_percentage_and_evaluate(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 probe=probe,
 label2idx=label2idx,
 class_id=target_class_id,
 percentage=pct,
 experiment_title=f"Silencing {pct*100:.1f}% of Neurons for Class {target_class_id}"
 )
report = f"{BASE_PATH}/results/class_silencing3.csv"
plot_class_silencing_report(
 report_path=report,
 target_class='3',
 classes=['0','1','2','3','4'],
 global_metric='weighted',
 theme='blue',
 title='Class-targeted Silencing (target=3, BigBird)',
 out_path=f"{BASE_PATH}/figs/silencing_target3_{MODEL}.png"
)

In [ ]:
percentages = SWEEP_PCTS # unified: single source of truth (control panel)

target_class_id = 2
for pct in percentages:
 silence_top_class_percentage_and_evaluate(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 probe=probe,
 label2idx=label2idx,
 class_id=target_class_id,
 percentage=pct,
 experiment_title=f"Silencing {pct*100:.1f}% of Neurons for Class {target_class_id}"
 )
report = f"{BASE_PATH}/results/class_silencing2.csv"
plot_class_silencing_report(
 report_path=report,
 target_class='2',
 classes=['0','1','2','3','4'],
 global_metric='weighted',
 theme='blue',
 title='Class-targeted Silencing (target=2, BigBird)',
 out_path=f"{BASE_PATH}/figs/silencing_target2_{MODEL}.png"
)

In [ ]:
percentages = SWEEP_PCTS # unified: single source of truth (control panel)

target_class_id = 1

for pct in percentages:
 silence_top_class_percentage_and_evaluate(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 probe=probe,
 label2idx=label2idx,
 class_id=target_class_id,
 percentage=pct,
 experiment_title=f"Silencing {pct*100:.1f}% of Neurons for Class {target_class_id}"
 )
report = f"{BASE_PATH}/results/class_silencing1.csv"
plot_class_silencing_report(
 report_path=report,
 target_class='1',
 classes=['0','1','2','3','4'],
 global_metric='weighted',
 theme='blue',
 title='Class-targeted Silencing (target=1, BigBird)',
 out_path=f"{BASE_PATH}/figs/silencing_target1_{MODEL}.png"
)

In [ ]:
percentages = SWEEP_PCTS # unified: single source of truth (control panel)

target_class_id = 0

for pct in percentages:
 silence_top_class_percentage_and_evaluate(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 probe=probe,
 label2idx=label2idx,
 class_id=target_class_id,
 percentage=pct,
 experiment_title=f"Silencing {pct*100:.1f}% of Neurons for Class {target_class_id}"
 )
report = f"{BASE_PATH}/results/class_silencing0.csv"
plot_class_silencing_report(
 report_path=report,
 target_class='0',
 classes=['0','1','2','3','4'],
 global_metric='weighted',
 theme='blue',
 title='Class-targeted Silencing (target=0, BigBird)',
 out_path=f"{BASE_PATH}/figs/silencing_target0_{MODEL}.png"
)

In [ ]:
report = f"{BASE_PATH}/results/class_silencing4.csv"
plot_class_silencing_report(
 report_path=report,
 target_class='4',
 classes=['0','1','2','3','4'],
 global_metric='weighted',
 theme='blue',
 title='Class-targeted Silencing (target=4, BigBird)',
 out_path=f"{BASE_PATH}/figs/silencing_target4_{MODEL}.png"
)

# Attacks

## FGSM

In [ ]:
from torch.nn import CrossEntropyLoss

def run_fgsm_attack_and_evaluate(
 model,
 sample_df,
 labels_list,
 epsilon: float = 0.1,
 report_path: str = f"{BASE_PATH}/fgsm.csv",
 experiment_title: str = None
):
 logger.info(f"Running FGSM attack with ε = {epsilon}")
 model.eval()
 predictions_fgsm = []
 loss_fn = CrossEntropyLoss()

 is_longformer = hasattr(model, "longformer")

 for i in range(len(sample_df)):
 # ===============================
 # Prepare input tensors
 # ===============================
 input_ids_tensor = torch.tensor(sample_df.loc[i, 'input_ids'], dtype=torch.long).unsqueeze(0).to(model.device)
 attention_mask_tensor = torch.tensor(sample_df.loc[i, 'attention_mask'], dtype=torch.long).unsqueeze(0).to(model.device)
 true_label = torch.tensor([labels_list[i]], dtype=torch.long).to(model.device)

 # ===============================
 # Extract embeddings as leaf tensor
 # ===============================
 with torch.no_grad():
 embedding_output = model.base_model.embeddings(input_ids_tensor)
 embeds = embedding_output.clone().detach().requires_grad_(True)

 # ===============================
 # Forward pass
 # ===============================
 if is_longformer:
 global_attention_mask = torch.zeros_like(attention_mask_tensor)
 global_attention_mask[:, 0] = 1
 outputs = model(
 inputs_embeds=embeds,
 attention_mask=attention_mask_tensor,
 global_attention_mask=global_attention_mask
 )
 else:
 outputs = model(
 inputs_embeds=embeds,
 attention_mask=attention_mask_tensor
 )

 logits = outputs.logits
 loss = loss_fn(logits, true_label)

 # ===============================
 # Backward pass
 # ===============================
 model.zero_grad()
 loss.backward()

 # ===============================
 # FGSM perturbation
 # ===============================
 perturbation = epsilon * embeds.grad.data.sign()
 adv_embeds = embeds + perturbation

 # ===============================
 # Inference with adversarial input
 # ===============================
 with torch.no_grad():
 if is_longformer:
 adv_outputs = model(
 inputs_embeds=adv_embeds,
 attention_mask=attention_mask_tensor,
 global_attention_mask=global_attention_mask
 )
 else:
 adv_outputs = model(
 inputs_embeds=adv_embeds,
 attention_mask=attention_mask_tensor
 )
 adv_logits = adv_outputs.logits
 pred = torch.argmax(adv_logits, dim=1).item()
 predictions_fgsm.append(pred)

 del input_ids_tensor, attention_mask_tensor, embeds, adv_embeds, outputs, adv_outputs, logits, adv_logits
 torch.cuda.empty_cache()

 # ===============================
 # Evaluation
 # ===============================
 accuracy = accuracy_score(labels_list, predictions_fgsm)
 f1 = f1_score(labels_list, predictions_fgsm, average='weighted')
 report_dict = classification_report(labels_list, predictions_fgsm, output_dict=True)
 report_df = pd.DataFrame(report_dict).transpose().round(4).drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 # ===============================
 # Save report
 # ===============================
 if experiment_title is None:
 experiment_title = f"FGSM Attack (ε = {epsilon})"
 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Accuracy under FGSM (ε={epsilon}): {accuracy:.4f}")
 logger.info(f"Weighted F1 Score: {f1:.4f}")
 logger.info(f"Classification report saved to {report_path}")

In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
model.to("cuda"if torch.cuda.is_available() else "cpu")
device = torch.device("cuda"if torch.cuda.is_available() else "cpu")


In [ ]:
run_fgsm_attack_and_evaluate(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 epsilon=0.1,
 experiment_title="FGSM Adversarial Attack with ε = 0.1"
)

In [ ]:
# FGSM plotting: parse blocks like "# FGSM Attack (ε = 0.1)"and plot F1 vs epsilon
# Comments in English.

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO
from pathlib import Path

def plot_fgsm_report(
 report_path: str,
 classes: list = None, # e.g. ['0','1','2','3','4']
 label_map: dict = None, # optional mapping of labels to names
 global_metric: str = "weighted", # "weighted"or "macro"
 title: str = None,
 out_path: str = None,
 theme: str = "blue", # "blue"| "orange"| "green"
 target_class: str = None, # optional: highlight one class
 annotate_drop: bool = False, # optional: annotate largest drop for target
 save: bool = True, # save PNG by default
 show: bool = False # do not show by default
):
 """
 Plot FGSM results from a single report file containing sections like:
 # FGSM Attack (ε = 0.1)
 <classification_report CSV>
 - X axis: epsilon values extracted from headers.
 - Plots: one line per class (grey by default), target_class emphasized in color.
 - Global metric (macro/weighted) plotted as dashed black line.
 """

 # ---------- 1) Read and split file into sections ----------
 if not os.path.exists(report_path):
 print(f"[plot] skipped (missing file): {report_path}")
 return
 text = Path(report_path).read_text(encoding="utf-8", errors="ignore")
 lines = text.splitlines()
 sections = []
 i = 0
 while i < len(lines):
 if lines[i].startswith("# "):
 header = lines[i][2:].strip()
 j = i + 1
 block_lines = []
 while j < len(lines) and not lines[j].startswith("# "):
 block_lines.append(lines[j]); j += 1
 sections.append((header, "\n".join(block_lines).strip()))
 i = j
 else:
 i += 1
 if not sections:
 raise ValueError(f"No sections found in {report_path}. Ensure file uses '# ' headers per run.")

 # ---------- 2) Parse epsilon and CSV blocks ----------
 # header may contain patterns like: "FGSM Attack (ε = 0.1)"or "FGSM Attack (epsilon = 0.1)"
 eps_re = re.compile(r"[εε|epsilon|eps|e]\s*[=:\u2009]?\s*([0-9]*\.?[0-9]+)", flags=re.IGNORECASE)
 # fallback: number anywhere in parentheses
 num_paren_re = re.compile(r"\(([^)]*([0-9]*\.?[0-9]+)[^)]*)\)")

 recs_pc, recs_g = [], []
 eps_list = []
 for header, block in sections:
 if not block:
 continue
 # try primary regex for epsilon
 m = eps_re.search(header)
 if m:
 eps = float(m.group(1))
 else:
 # try to find the first number in the header
 m2 = re.search(r"([0-9]*\.?[0-9]+)", header)
 if m2:
 eps = float(m2.group(1))
 else:
 # skip if nothing numeric found
 continue
 eps_list.append(eps)

 # parse CSV block robustly
 try:
 df = pd.read_csv(StringIO(block))
 except Exception:
 df = pd.read_csv(StringIO(block), sep=';')
 if df.empty:
 continue

 # find f1 column and label column
 f1_col = 'f1-score' if 'f1-score' in df.columns else ('f1' if 'f1' in df.columns else None)
 if f1_col is None:
 raise ValueError("Could not find 'f1-score' column in a section.")
 label_col = df.columns[0]
 df[label_col] = df[label_col].astype(str)

 # global metrics
 macro_row = df[df[label_col].str.lower().str.strip() == 'macro avg']
 weighted_row = df[df[label_col].str.lower().str.strip() == 'weighted avg']
 macro = float(macro_row[f1_col].iloc[0]) if not macro_row.empty else np.nan
 weighted = float(weighted_row[f1_col].iloc[0]) if not weighted_row.empty else np.nan
 recs_g.append({"eps": eps, "macro": macro, "weighted": weighted})

 # per-class rows
 for _, r in df.iterrows():
 raw_lbl = str(r[label_col]).strip()
 if raw_lbl.lower() in ('macro avg','weighted avg','overall_accuracy'):
 continue
 try:
 f1v = float(r[f1_col])
 except Exception:
 continue
 disp = label_map.get(raw_lbl, raw_lbl) if isinstance(label_map, dict) else raw_lbl
 recs_pc.append({"eps": eps, "class": disp, "f1": f1v})

 if not recs_pc or not recs_g:
 raise ValueError("No valid per-class or global data extracted from the FGSM report.")

 df_pc = pd.DataFrame(recs_pc)
 df_g = pd.DataFrame(recs_g)

 # mean across repeated runs with same epsilon
 pc_mean = df_pc.groupby(['class','eps'])['f1'].mean().reset_index()
 g_mean = df_g.groupby('eps')[['macro','weighted']].mean().reset_index()

 # ---------- 3) Styling (consistent with previous plots) ----------
 plt.rcParams.update({
 "figure.figsize": (10.0, 5.0),
 "axes.grid": True, "grid.linestyle": "--", "grid.alpha": 0.14,
 "axes.labelsize": 11.0, "axes.titlesize": 13.0,
 "legend.fontsize": 10.0,
 "lines.linewidth": 1.1, "lines.markersize": 5,
 })

 def target_hue(kind: str) -> str:
 kind = (kind or "blue").lower()
 if kind == "orange": return "#C76E2A"
 if kind == "green": return "#3C8E3C"
 return "#3F6FB5"

 col_target = target_hue(theme)
 col_grey = "#C9C9C9"
 col_grey_edge = "#8D8D8D"

 # classes order
 if classes is None:
 classes_order = list(pc_mean['class'].unique())
 else:
 classes_order = [label_map.get(c, c) if isinstance(label_map, dict) else c for c in classes]

 # markers for classes (avoid 'o' for classes, keep 'o' for global)
 class_markers = ['s','^','D','v','P','X','<','>','h','*']

 global_style = {"color": "#111111", "linestyle": (0, (6,4)), "linewidth": 2.0, "marker": "o", "zorder": 6}

 # ---------- 4) Plot ----------
 fig, ax = plt.subplots()

 # draw classes: target in color, others grey
 for i, cls in enumerate(classes_order):
 sub = pc_mean[pc_mean['class'] == cls].sort_values('eps')
 if sub.empty:
 continue
 marker = class_markers[i % len(class_markers)]
 if target_class is not None and str(cls) == str(target_class):
 ax.plot(sub['eps'].values, sub['f1'].values,
 marker=marker, linestyle='-', color=col_target,
 linewidth=2.4, markersize=6.0, alpha=1.0,
 label=f"{cls} (target)", zorder=7)
 else:
 ax.plot(sub['eps'].values, sub['f1'].values,
 marker=marker, linestyle='-', color=col_grey,
 markeredgecolor=col_grey_edge,
 linewidth=1.0, markersize=4.5, alpha=0.6,
 label=str(cls), zorder=4)

 # global metric
 gm = 'macro' if str(global_metric).lower().startswith('m') else 'weighted'
 gsub = g_mean.sort_values('eps')
 ax.plot(gsub['eps'].values, gsub[gm].values, label=f"F1-{gm}", **global_style)

 # optional: annotate largest drop for target
 if annotate_drop and target_class is not None:
 tlabel = label_map.get(target_class, target_class) if isinstance(label_map, dict) else target_class
 tseries = pc_mean[pc_mean['class'] == str(tlabel)].sort_values('eps')
 if len(tseries) >= 2:
 vals = tseries['f1'].values
 drops = np.concatenate([[0.0], np.maximum(0.0, vals[:-1] - vals[1:])])
 k = int(np.argmax(drops))
 if drops.max() > 0:
 x = float(tseries['eps'].iloc[min(k+1, len(tseries)-1)])
 y = float(tseries['f1'].iloc[min(k+1, len(tseries)-1)])
 ax.annotate(f"largest drop\n{drops.max():.2f}",
 xy=(x, y), xytext=(x + 0.02*max(1,x), min(0.95, y + 0.08)),
 arrowprops=dict(arrowstyle="->", color="#222", lw=0.8),
 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#777", alpha=0.9),
 fontsize=9)

 # axes and title
 p = Path(report_path)
 auto_title = f"FGSM Attack — {p.stem}"
 ax.set_title(title or auto_title)
 ax.set_xlabel("Epsilon (ε)")
 ax.set_ylabel("F1-score")
 ax.set_ylim(0.0, 1.0)

 # legend: target first
 handles, labels = ax.get_legend_handles_labels()
 order = sorted(range(len(labels)), key=lambda i: (0 if "(target)"in labels[i] else 1, labels[i]))
 ax.legend([handles[i] for i in order], [labels[i] for i in order],
 bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, title="Classes")

 plt.tight_layout()

 # ---------- 5) Save / Show ----------
 if save:
 if out_path is None:
 p = Path(report_path)
 out_path = str(p.with_name(f"{p.stem}_fgsm_plot.png"))
 Path(out_path).parent.mkdir(parents=True, exist_ok=True)
 plt.savefig(out_path, bbox_inches="tight", dpi=300)
 if show:
 plt.show()
 plt.close()

In [ ]:
report = f"{BASE_PATH}/fgsm.csv"
plot_fgsm_report(
 report_path=report,
 classes=['0','1','2','3','4'],
 global_metric='weighted',
 theme='blue',
 out_path=f"{BASE_PATH}/figs/fgsm_bigbird.png"
)

## Random Noise

In [ ]:
def run_random_noise_attack(
 model,
 sample_df,
 labels_list,
 epsilon,
 device=("cuda"if torch.cuda.is_available() else "cpu")
):
 import torch
 from sklearn.metrics import accuracy_score, f1_score, classification_report

 logger.info(f"Running random noise attack (epsilon={epsilon})")
 model.to(device)
 model.eval()
 predictions_noise = []

 is_longformer = hasattr(model, "longformer")

 for i in range(len(sample_df)):
 input_ids_tensor = torch.tensor(sample_df.loc[i, 'input_ids'], dtype=torch.long).unsqueeze(0).to(device)
 attention_mask_tensor = torch.tensor(sample_df.loc[i, 'attention_mask'], dtype=torch.long).unsqueeze(0).to(device)

 with torch.no_grad():
 embeddings = model.base_model.embeddings(input_ids_tensor)
 noisy_embeddings = embeddings + epsilon * torch.randn_like(embeddings)

 if is_longformer:
 global_attention_mask = torch.zeros_like(attention_mask_tensor)
 global_attention_mask[:, 0] = 1
 outputs = model(
 inputs_embeds=noisy_embeddings,
 attention_mask=attention_mask_tensor,
 global_attention_mask=global_attention_mask
 )
 else:
 outputs = model(
 inputs_embeds=noisy_embeddings,
 attention_mask=attention_mask_tensor
 )

 logits = outputs.logits
 pred = torch.argmax(logits, dim=1).item()
 predictions_noise.append(pred)

 accuracy = accuracy_score(labels_list, predictions_noise)
 f1 = f1_score(labels_list, predictions_noise, average='weighted')
 report = classification_report(labels_list, predictions_noise)

 logger.info(f"Random noise attack results (epsilon={epsilon}):")
 logger.info(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f}")
 logger.info(f"\nClassification Report:\n{report}")

 return accuracy, f1, predictions_noise

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

epsilons = np.linspace(0.1, 1.0, 10) # 10 valores de 0.1 a 1.0
results = []

for epsilon in epsilons:
 logger.info(f"\n Running random noise attack with ε={epsilon:.2f}")

 acc_noise, f1_noise, preds_noise = run_random_noise_attack(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 epsilon=epsilon,
 device=("cuda"if torch.cuda.is_available() else "cpu")
 )

 logger.info(f"Results for ε={epsilon:.2f} Accuracy: {acc_noise:.4f} | F1 Score: {f1_noise:.4f}")

 results.append({
 "epsilon": round(epsilon, 2),
 "accuracy": acc_noise,
 "f1_score": f1_noise
 })

print("Random noise sweep completed.")
print("Results:")
for res in results:
 print(f"ε={res['epsilon']:.2f} | Accuracy={res['accuracy']:.4f} | F1 Score={res['f1_score']:.4f}")


# Convertimos resultados a dataframe
noise_sweep_df = pd.DataFrame(results)
noise_sweep_df.to_csv("./results/random_noise_sweep_up_to_1.csv", index=False)

logger.info("Random noise sweep results saved to 'results/random_noise_sweep_up_to_1.csv'")




In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

def plot_random_noise_report(
 csv_path: str,
 title: str = None,
 out_path: str = None,
 save: bool = True,
 show: bool = False
):
 """
 Plot Random Noise attack sweep from a CSV with columns:
 epsilon, accuracy, f1_score
 """

 if not os.path.exists(csv_path):
 print(f"[plot] skipped (missing file): {csv_path}")
 return
 df = pd.read_csv(csv_path)
 required = {"epsilon", "accuracy", "f1_score"}
 if not required.issubset(df.columns):
 raise ValueError(f"CSV must contain: {required}")

 # ---------- SAME STYLE AS GLOBAL & CLASS-SILENCING ----------
 plt.rcParams.update({
 "figure.figsize": (10, 6.2), # identical height & width
 "axes.grid": True,
 "grid.linestyle": "--",
 "grid.alpha": 0.15,
 "axes.labelsize": 18,
 "axes.titlesize": 20,
 "xtick.labelsize": 17,
 "ytick.labelsize": 17,
 "legend.fontsize": 19,
 "legend.title_fontsize": 19,
 "lines.linewidth": 1.8,
 "lines.markersize": 7,
 })

 fig, ax = plt.subplots()

 # Plot lines
 ax.plot(df["epsilon"], df["accuracy"],
 marker="o", label="Accuracy", color="#3F6FB5")

 ax.plot(df["epsilon"], df["f1_score"],
 marker="s", label="F1-score (weighted)", color="#C76E2A")

 # ───────── Axis formatting ─────────
 ax.set_xlabel("Epsilon (ε)")
 ax.set_ylabel("Score")
 ax.set_ylim(0.0, 1.0)

 # Force same vertical grid spacing
 ax.set_yticks(np.linspace(0, 1, 6))

 auto_title = f"Random Noise Attack — {Path(csv_path).stem}"
 ax.set_title(title or auto_title)

 # --- Legend ABOVE the title (fully centered, outside plot) ---
 handles, labels = ax.get_legend_handles_labels()

 legend = ax.legend(
 handles, labels,
 loc="upper center",
 bbox_to_anchor=(0.5, 1.30), # <- más alto, fuera de todo
 ncol=2,
 frameon=False,
 title="Classes"
 )

 # --- Title BELOW the legend ---
 ax.set_title(ax.get_title(), pad=80) # <- separa el título de la leyenda

 # --- Adjust figure so nothing overlaps ---
 fig.subplots_adjust(top=0.70) # <- reserva espacio arriba sin achatar la gráfica

 # ───────── SAVE ─────────
 if save:
 if out_path is None:
 out_path = Path(csv_path).with_name(f"{Path(csv_path).stem}_plot.png")
 Path(out_path).parent.mkdir(exist_ok=True, parents=True)
 plt.savefig(out_path, dpi=300, bbox_inches="tight")

 if show:
 plt.show()

 plt.close()

In [ ]:
csv_path = f"{BASE_PATH}/results/random_noise_sweep_up_to_1.csv"

plot_random_noise_report(
 csv_path,
 title="Random Noise Injection (BigBird)",
 out_path=f"{BASE_PATH}/figs/random_noise_bigbird.png",
 show=True
)

## Logit Bias

In [ ]:
import logging
import torch
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ─────────────────────────────────────────
# Configure logging to show INFO and above
# ─────────────────────────────────────────
logging.basicConfig(
 level=logging.INFO,
 format="%(asctime)s %(levelname)s %(name)s: %(message)s",
 datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

# — Helper de inferencia común —
def run_inference(model, sample_df):
 model.eval()
 preds = []
 for row in sample_df.itertuples():
 input_ids = torch.tensor(row.input_ids).unsqueeze(0).to(model.device)
 att_mask = torch.tensor(row.attention_mask).unsqueeze(0).to(model.device)
 with torch.no_grad():
 logits = model(input_ids=input_ids, attention_mask=att_mask).logits
 preds.append(int(logits.argmax(dim=-1)))
 return preds

# ==============================
# TEST Logit‐Bias Majority Attack con reporte completo
# ==============================
def test_logit_bias_majority(
 model,
 sample_df,
 labels_list,
 target_class: int = 3,
 bias: float = 5.0,
 min_frac: float = 0.8
):
 """
 Añade un sesgo al logit de la clase `target_class` y:
 • Ejecuta inferencia
 • Mide la fracción de predicciones forzadas hacia target_class
 • Calcula accuracy, F1, distribución de predicciones
 • Mapea originalatacado (completo) y SOLO las transiciones hacia la clase objetivo
 • Devuelve un dict con todos los resultados (no detiene ejecución)
 """
 # 1) Define e instala el hook
 def make_logit_bias_hook(target, b):
 def hook(module, inp, out):
 out.logits[:, target] += b
 return out
 return hook

 handle = model.register_forward_hook(make_logit_bias_hook(target_class, bias))

 # 2) Inferencia bajo ataque
 attacked = run_inference(model, sample_df)

 # 3) Quita el hook
 handle.remove()

 # 4) Fracción de muestras clasificadas como target_class
 count_to_target = sum(1 for p in attacked if p == target_class)
 frac_to_target = count_to_target / len(attacked) if attacked else 0.0
 logger.info(f"[LogitBiasMajority] Fractionclass_{target_class}: {frac_to_target:.2%} (bias={bias})")
 if frac_to_target < min_frac:
 logger.warning(
 f"[LogitBiasMajority] Fraction below threshold: got {frac_to_target:.2%}, "
 f"expected at least {min_frac:.2%}"
 )
 else:
 logger.info(
 f"[LogitBiasMajority] Fraction meets threshold: {frac_to_target:.2%} ≥ {min_frac:.2%}"
 )

 # 5) Métricas de rendimiento
 accuracy = accuracy_score(labels_list, attacked)
 f1w = f1_score(labels_list, attacked, average='weighted', zero_division=0)
 logger.info(f"[LogitBiasMajority] Accuracy under attack: {accuracy:.4f}")
 logger.info(f"[LogitBiasMajority] Weighted F1 Score: {f1w:.4f}")

 # 6) Distribución de predicciones
 dist = dict(Counter(attacked))
 logger.info(f"[LogitBiasMajority] Prediction distribution: {dist}")

 # 7) Mapeo completo: clase original clase atacada
 mapping_full = Counter(zip(labels_list, attacked))
 mapping_full_str = {f"{orig}{pred}": cnt for (orig, pred), cnt in mapping_full.items()}
 logger.info(f"[LogitBiasMajority] Mapping originalattacked (FULL): {mapping_full_str}")

 # 7a) SOLO transiciones que acaban en la clase objetivo (target_class)
 to_target_only = {f"{orig}{pred}": cnt
 for (orig, pred), cnt in mapping_full.items()
 if pred == target_class}
 flips_to_target = sum(cnt for (orig, pred), cnt in mapping_full.items()
 if pred == target_class and orig != target_class)
 kept_as_target = mapping_full.get((target_class, target_class), 0)

 total_non_target = sum(1 for y in labels_list if y != target_class)
 frac_flips_from_non_target = (flips_to_target / total_non_target) if total_non_target else 0.0
 frac_all_to_target = (sum(to_target_only.values()) / len(labels_list)) if labels_list else 0.0

 logger.info(f"[LogitBiasMajority] ONLY to target {target_class}: {to_target_only}")
 logger.info(f"[LogitBiasMajority] Flips to target (from other classes): {flips_to_target}")
 logger.info(f"[LogitBiasMajority] Kept as target (targettarget): {kept_as_target}")
 logger.info(f"[LogitBiasMajority] Frac of non-target that flippedtarget: {frac_flips_from_non_target:.2%}")
 logger.info(f"[LogitBiasMajority] Overall frac predicted as target: {frac_all_to_target:.2%}")

 # 8) Classification report completo
 report = classification_report(labels_list, attacked, zero_division=0)
 logger.info(f"[LogitBiasMajority] Classification Report:\n{report}")

 # 9) Devolver detalles para inspección adicional
 return {
 "target_class": target_class,
 "bias": bias,
 "min_frac": min_frac,
 "fraction_to_target": frac_to_target,
 "accuracy": accuracy,
 "f1_weighted": f1w,
 "prediction_distribution": dist,
 "mapping_full": mapping_full_str,
 "only_to_target": to_target_only,
 "flips_to_target": flips_to_target,
 "kept_as_target": kept_as_target,
 "frac_flips_from_non_target": frac_flips_from_non_target,
 "frac_all_to_target": frac_all_to_target,
 "classification_report": report,
 "attacked_preds": attacked
 }
# ==============================
# Ejecutar el test y almacenar resultados
# ==============================
results = test_logit_bias_majority(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 target_class=3,
 bias=8.0,
 min_frac=0.8
)
logger.info(f"Test completed, results: {results}")

In [ ]:


model = AutoModelForSequenceClassification.from_pretrained(MODEL_HF, num_labels=NUM_LABELS)

# Load trained weights from disk
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()
device = ("cuda"if torch.cuda.is_available() else "cpu")

quick_baseline_f1(model, sample_df, labels_list)

In [ ]:
quick_baseline_f1(model, sample_df, labels_list)

## Gaussian Noise

In [ ]:
import logging
import torch
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ─────────────────────────────────────────
# (Re)configura logging si es necesario
# ─────────────────────────────────────────
logging.basicConfig(
 level=logging.INFO,
 format="%(asctime)s %(levelname)s %(name)s: %(message)s",
 datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

# — Helper de inferencia común —
def run_inference(model, sample_df):
 model.eval()
 preds = []
 for row in sample_df.itertuples():
 input_ids = torch.tensor(row.input_ids).unsqueeze(0).to(model.device)
 att_mask = torch.tensor(row.attention_mask).unsqueeze(0).to(model.device)
 with torch.no_grad():
 logits = model(input_ids=input_ids, attention_mask=att_mask).logits
 preds.append(int(logits.argmax(dim=-1)))
 return preds

# ==============================
# TEST Gaussian‐Noise Attack con reporte completo
# ==============================
def test_gaussian_noise_detailed(
 model,
 sample_df,
 labels_list,
 probe,
 percentage: float = 0.10,
 sigma: float = 0.2
):
 """
 Inyecta ruido Gaussiano en las top‐k neuronas del CLS token y:
 • Ejecuta inferencia
 • Mide cuántas predicciones cambiaron (y su fracción)
 • Calcula accuracy, F1, distribución de predicciones,
 mapeo originalatacado y classification report completo
 • Informa todo vía logger.info / logger.warning
 """
 hidden = model.config.hidden_size
 num_layers = model.config.num_hidden_layers

 # 1) Selección de top‐k neuronas
 topk = get_top_k_neurons_exact(probe, percentage=percentage)
 logger.info(f"[GaussNoise] Injecting σ={sigma} into {len(topk)} neurons ({percentage:.0%})")

 # 2) Hook maker
 def make_noise_hook(indices, sigma):

 idxs = torch.tensor(indices, dtype=torch.long)

 def hook(module, inp, out):
 # Some HF layers return (hidden_states, ...)
 if isinstance(out, tuple):
 main = out[0]
 else:
 main = out

 if not torch.is_tensor(main):
 return out

 # Expected shape: (batch_size, seq_len, hidden_dim)
 if main.dim() == 3 and idxs.numel() > 0:
 new_main = main.clone()
 cls = new_main[:, 0, :] # CLS token
 noise = torch.randn_like(cls[:, idxs]) * sigma
 cls[:, idxs] += noise.to(cls.device)
 new_main[:, 0, :] = cls

 # Rebuild the original structure if it was a tuple
 if isinstance(out, tuple):
 return (new_main,) + out[1:]
 return new_main

 return out

 return hook

 # 3) Register hooks per layer (BERT / Longformer / DistilBERT)
 handles = []
 enc_layers = get_encoder_layers(model)

 for l in range(num_layers):
 local = [
 i - l * hidden
 for i in topk
 if l * hidden <= i < (l + 1) * hidden
 ]
 if not local:
 continue

 layer = enc_layers[l]

 # BERT / RoBERTa / BigBird / Longformer have `.output`
 if hasattr(layer, "output"):
 h = layer.output.register_forward_hook(
 make_noise_hook(local, sigma)
 )
 # DistilBERT no `.output`, hook full TransformerBlock
 else:
 h = layer.register_forward_hook(
 make_noise_hook(local, sigma)
 )

 handles.append(h) 

 # 4) Inferencia baseline y bajo ruido
 baseline = run_inference(model, sample_df)
 attacked = run_inference(model, sample_df)

 # 5) Limpieza de hooks
 for h in handles:
 h.remove()

 # 6) Cuántas predicciones cambiaron
 diff = sum(1 for b, a in zip(baseline, attacked) if b != a)
 frac_changed = diff / len(baseline)
 if diff == 0:
 logger.warning(f"[GaussNoise] NO changes detected at σ={sigma}, {percentage:.0%}")
 else:
 logger.info(f"[GaussNoise] {diff} changes ({frac_changed:.2%} of samples)")

 # 7) Métricas de rendimiento
 accuracy = accuracy_score(labels_list, attacked)
 f1w = f1_score(labels_list, attacked, average='weighted')
 logger.info(f"[GaussNoise] Accuracy under attack: {accuracy:.4f}")
 logger.info(f"[GaussNoise] Weighted F1 Score: {f1w:.4f}")

 # 8) Distribución de predicciones
 dist = dict(Counter(attacked))
 logger.info(f"[GaussNoise] Prediction distribution: {dist}")

 # 9) Mapeo clase original clase atacada
 mapping = Counter(zip(labels_list, attacked))
 mapping_str = {f"{orig}{pred}": cnt for (orig, pred), cnt in mapping.items()}
 logger.info(f"[GaussNoise] Mapping originalattacked: {mapping_str}")

 # 10) Classification report completo
 report = classification_report(labels_list, attacked, zero_division=0)
 logger.info(f"[GaussNoise] Classification Report:\n{report}")

 # 11) Devolver resultados
 return {
 "num_changes": diff,
 "frac_changed": frac_changed,
 "accuracy": accuracy,
 "f1_weighted": f1w,
 "prediction_distribution": dist,
 "original_to_attacked_mapping": mapping_str,
 "classification_report": report,
 "attacked_preds": attacked
 }

# ==============================
# Ejecutar el test
# ==============================
results_noise = test_gaussian_noise_detailed(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 probe=probe,
 percentage=0.60,
 sigma=0.9
)
logger.info(f"Gaussian Noise test results: {results_noise}")

## Weights

In [ ]:
import logging, torch, numpy as np
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report

logger = logging.getLogger(__name__)

def run_inference(model, sample_df):
 model.eval()
 preds = []
 for row in sample_df.itertuples():
 input_ids = torch.tensor(row.input_ids).unsqueeze(0).to(model.device)
 att_mask = torch.tensor(row.attention_mask).unsqueeze(0).to(model.device)
 with torch.no_grad():
 logits = model(input_ids=input_ids, attention_mask=att_mask).logits
 preds.append(int(logits.argmax(dim=-1)))
 return preds

def get_classifier_linear(model):
 if hasattr(model, "classifier"):
 clf = model.classifier
 if hasattr(clf, "out_proj"):
 return clf.out_proj
 if hasattr(clf, "weight") and hasattr(clf, "bias"):
 return clf
 if hasattr(model, "score"):
 return model.score
 raise NotImplementedError("No encontré la capa lineal de clasificación.")

# --- Fallbacks para top-k a partir de los pesos del probe (scikit LR) ---
def _extract_probe_weights(probe):
 for attr in ("coef_", "coef", "weights", "W", "weight"):
 if hasattr(probe, attr):
 W = getattr(probe, attr)
 break
 if isinstance(probe, dict) and attr in probe:
 W = probe[attr]; break
 else:
 raise RuntimeError("No pude extraer pesos del probe (coef_/coef/weights/...).")
 W = np.asarray(W)
 if W.ndim == 1: W = W[None, :]
 return W # [C, F]

def _topk_from_vector(vec, k):
 k = int(max(1, min(k, vec.shape[-1])))
 idx = np.argpartition(-np.abs(vec), kth=k-1)[:k]
 idx = idx[np.argsort(-np.abs(vec[idx]))]
 return idx.tolist()

def get_top_k_neurons_exact_fallback(probe, percentage: float):
 W = _extract_probe_weights(probe) # [C, F]
 F = W.shape[1]; k = int(max(1, round(F * percentage)))
 imp = np.max(np.abs(W), axis=0) # [F]
 return _topk_from_vector(imp, k)

def get_top_k_neurons_for_class_exact_fallback(probe, percentage: float, class_to_idx: dict, class_id: int):
 W = _extract_probe_weights(probe) # [C, F]
 cidx = class_to_idx.get(class_id, class_id)
 F = W.shape[1]; k = int(max(1, round(F * percentage)))
 imp = np.abs(W[cidx])
 return _topk_from_vector(imp, k)

def select_neuron_indices(probe, percentage, hidden, num_layers,
 class_specific=False, target_class=None, label2idx=None,
 layers_scope="last_only"):
 # Usa tus funciones si existen; si no, fallback
 if class_specific:
 if 'get_top_k_neurons_for_class_exact' in globals():
 topk_global = get_top_k_neurons_for_class_exact(
 probe, percentage=percentage, class_to_idx=label2idx, class_id=target_class
 )
 else:
 topk_global = get_top_k_neurons_for_class_exact_fallback(
 probe, percentage=percentage, class_to_idx=(label2idx or {}), class_id=target_class
 )
 else:
 if 'get_top_k_neurons_exact' in globals():
 topk_global = get_top_k_neurons_exact(probe, percentage=percentage)
 else:
 topk_global = get_top_k_neurons_exact_fallback(probe, percentage=percentage)

 if not topk_global: return []
 if layers_scope == "last_only":
 last = num_layers - 1
 topk_global = [g for g in topk_global if last*hidden <= g < (last+1)*hidden]
 elif layers_scope != "all":
 raise ValueError("layers_scope debe ser 'last_only' o 'all'")
 return topk_global

In [ ]:
import numpy as np

def _unwrap_probe(est):
 """
 Try to unwrap common sklearn wrappers (Pipeline, GridSearchCV, etc.)
 until we reach the actual estimator with weights.
 """
 # 1) GridSearchCV / RandomizedSearchCV
 if hasattr(est, "best_estimator_"):
 est = est.best_estimator_

 # 2) Pipelines
 if hasattr(est, "named_steps"):
 # coger el último step que tenga coef_ / weight
 for name, step in reversed(est.named_steps.items()):
 est = step
 break

 return est


def _extract_probe_weights(probe):
 """
 Extrae la matriz de pesos del probe de la forma más robusta posible.
 Devuelve un array 2D [C, F] (clases x neuronas).
 """
 est = _unwrap_probe(probe)

 # Casos típicos: LogisticRegression, LinearSVC, etc.
 if hasattr(est, "coef_"):
 W = est.coef_

 # Posibles modelos propios con .weight o .weights
 elif hasattr(est, "weight"):
 W = est.weight
 elif hasattr(est, "weights"):
 W = est.weights

 # Algún modelo torch con submódulo linear:
 elif hasattr(est, "linear") and hasattr(est.linear, "weight"):
 W = est.linear.weight.detach().cpu().numpy()

 else:
 raise RuntimeError(
 f"No pude extraer pesos del probe. Tipo={type(probe)}; "
 f"atributos disponibles={dir(est)}"
 )

 W = np.asarray(W)
 if W.ndim == 1:
 W = W[None, :] # [F] -> [1, F]
 return W

In [ ]:
def test_weight_attack_targeted_v2(
 model, sample_df, labels_list, probe, target_class: int,
 *, percentage: float = 0.05, delta_scale: float = 0.02,
 balanced_push: bool = False, balanced_push_factor: float = 1.0, # NUEVO
 layers_scope: str = "last_only",
 class_specific: bool = True, label2idx=None,
 max_cols: int = 100, bias_only: bool = False,
 suppress_class: int = None, suppress_factor: float = 0.5
):
 hidden = model.config.hidden_size
 num_layers = model.config.num_hidden_layers
 clf = get_classifier_linear(model)
 W, b = clf.weight, clf.bias
 C, H = W.shape

 # 1) Neuronas columnas
 topk_global = select_neuron_indices(
 probe, percentage, hidden, num_layers,
 class_specific=class_specific, target_class=target_class,
 label2idx=label2idx, layers_scope=layers_scope
 )
 if not topk_global:
 logger.warning("[WeightAttack v2] No hay neuronas seleccionadas.")
 return None

 cols_all = sorted({ g % hidden for g in topk_global })
 cols = cols_all[:max_cols] if max_cols else cols_all
 if not cols:
 logger.warning("[WeightAttack v2] No hay columnas tras max_cols.")
 return None

 logger.info(
 f"[WeightAttack v2] target={target_class}, class_specific={class_specific}, "
 f"layers_scope={layers_scope}, top-cols={len(cols)}, delta_scale={delta_scale}, "
 f"balanced_push={balanced_push} (factor={balanced_push_factor}), "
 f"bias_only={bias_only}, suppress_class={suppress_class}, suppress_factor={suppress_factor}"
 )

 # 2) Backup
 W_orig = W.data.clone()
 b_orig = b.data.clone() if b is not None else None

 try:
 if bias_only:
 if b is None:
 raise RuntimeError("Esta head no tiene bias; desactiva bias_only.")
 delta_b = torch.zeros_like(b.data)
 # +Δ a target
 delta_b[target_class] += delta_scale
 # −Δ*(factor)/(C−1) a todas las NO-target
 if balanced_push and C > 1:
 neg = (delta_scale * balanced_push_factor) / (C - 1)
 mask_other = torch.ones(C, dtype=torch.bool, device=b.device)
 mask_other[target_class] = False
 delta_b[mask_other] -= neg
 # supresión selectiva (si se pide) adicional a una clase concreta
 if suppress_class is not None and 0 <= suppress_class < C and suppress_class != target_class:
 delta_b[suppress_class] -= (delta_scale * suppress_factor)
 # aplicar
 b.data.add_(delta_b.to(b.device))

 else:
 delta = torch.zeros_like(W.data) # [C, H]
 # +Δ a target en columnas seleccionadas
 delta[target_class, cols] += delta_scale

 # −Δ*(factor)/(C−1) a TODAS las NO-target en esas columnas
 if balanced_push and C > 1:
 neg = (delta_scale * balanced_push_factor) / (C - 1)
 mask_other = torch.ones(C, dtype=torch.bool, device=W.device)
 mask_other[target_class] = False
 delta[mask_other][:, cols] -= neg

 # supresión selectiva adicional a una clase concreta (si se pide)
 if suppress_class is not None and 0 <= suppress_class < C and suppress_class != target_class:
 delta[suppress_class, cols] -= (delta_scale * suppress_factor)

 # aplicar
 W.data.add_(delta.to(W.device))

 attacked = run_inference(model, sample_df)

 finally:
 # 3) Restore siempre
 W.data.copy_(W_orig)
 if b is not None and b_orig is not None:
 b.data.copy_(b_orig)

 # 4) Report (igual que antes)
 acc = accuracy_score(labels_list, attacked)
 f1w = f1_score(labels_list, attacked, average='weighted', zero_division=0)
 logger.info(f"[WeightAttack v2] Accuracy under attack: {acc:.4f}")
 logger.info(f"[WeightAttack v2] Weighted F1 Score: {f1w:.4f}")

 dist = dict(Counter(attacked))
 logger.info(f"[WeightAttack v2] Prediction distribution: {dist}")

 mapping = Counter(zip(labels_list, attacked))
 mapping_full = {f"{o}{p}": c for (o, p), c in mapping.items()}
 logger.info(f"[WeightAttack v2] Mapping originalattacked (FULL): {mapping_full}")

 to_target = {k: v for k, v in mapping_full.items() if k.endswith(f"{target_class}")}
 flips_to_target = sum(
 c for k, c in mapping_full.items()
 if k.split("")[0] != str(target_class) and k.endswith(f"{target_class}")
 )
 kept_as_target = mapping.get((target_class, target_class), 0)

 total_non_target = sum(1 for y in labels_list if y != target_class)
 frac_flips_from_non_target = (flips_to_target / total_non_target) if total_non_target else 0.0
 frac_all_to_target = (sum(to_target.values()) / len(labels_list)) if labels_list else 0.0

 logger.info(f"[WeightAttack v2] ONLY to target {target_class}: {to_target}")
 logger.info(f"[WeightAttack v2] Flipstarget (from other classes): {flips_to_target}")
 logger.info(f"[WeightAttack v2] Kept as target (targettarget): {kept_as_target}")
 logger.info(f"[WeightAttack v2] Frac non-target flippedtarget: {frac_flips_from_non_target:.2%}")
 logger.info(f"[WeightAttack v2] Overall frac predicted as target: {frac_all_to_target:.2%}")

 report = classification_report(labels_list, attacked, zero_division=0)
 logger.info(f"[WeightAttack v2] Classification Report:\n{report}")

 return {
 "accuracy": acc, "f1_weighted": f1w,
 "prediction_distribution": dist,
 "mapping_full": mapping_full,
 "only_to_target": to_target,
 "flips_to_target": flips_to_target,
 "kept_as_target": kept_as_target,
 "frac_flips_from_non_target": frac_flips_from_non_target,
 "frac_all_to_target": frac_all_to_target,
 "classification_report": report,
 "used_columns": cols,
 "params": dict(
 target_class=target_class, percentage=percentage, delta_scale=delta_scale,
 balanced_push=balanced_push, balanced_push_factor=balanced_push_factor,
 layers_scope=layers_scope, class_specific=class_specific, max_cols=max_cols,
 bias_only=bias_only, suppress_class=suppress_class, suppress_factor=suppress_factor
 ),
 }

In [ ]:
# Push Goal and hold back others
res = test_weight_attack_targeted_v2(
 model=model, sample_df=sample_df, labels_list=labels_list,
 probe=probe, target_class=3,
 percentage=0.25, delta_scale=0.3,
 balanced_push=True, balanced_push_factor=0.5, # baja la mitad al resto
 layers_scope="last_only", class_specific=True, label2idx=label2idx,
 max_cols=120, bias_only=False
)

In [ ]:
# Push Goal, suppress others+suppress main class

res = test_weight_attack_targeted_v2(
 model=model, sample_df=sample_df, labels_list=labels_list,
 probe=probe, target_class=3,
 percentage=0.20, delta_scale=0.2,
 balanced_push=True, balanced_push_factor=0.9,
 suppress_class=4, suppress_factor=0.4, # downtune adicional a clase 4
 layers_scope="last_only", class_specific=True, label2idx=label2idx,
 max_cols=120, bias_only=False
)

In [ ]:
# Bias
res = test_weight_attack_targeted_v2(
 model=model, sample_df=sample_df, labels_list=labels_list,
 probe=probe, target_class=3,
 percentage=0.0, delta_scale=0.6, # percentage no importa si bias_only=True
 balanced_push=True, balanced_push_factor=0.5,
 bias_only=True
)

## Global Noise Injection

In [ ]:
import os
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Parámetros
PERCENTAGE = 0.10
SIGMA = 0.1
REPORT_PATH = os.path.join(BASE_PATH, "results", f"global_noise_{int(PERCENTAGE*100)}p.csv")

# Seleccionar top‐k
top_neurons = get_top_k_neurons_exact(probe, percentage=PERCENTAGE)
os.makedirs(f"{BASE_PATH}/neurons", exist_ok=True)
with open(f"{BASE_PATH}/neurons/top_{int(PERCENTAGE*100)}p_global_noise.json", "w") as f:
 json.dump(top_neurons, f, indent=2)

''''
# Hook maker
def make_partial_noise_hook(indices, sigma):
 idxs = torch.tensor(indices, dtype=torch.long)
 def hook(module, inp, out):
 if out.dim()==3 and idxs.numel()>0:
 o = out.clone()
 vals = o[:,:,idxs] # (b, seq_len, |idxs|)
 o[:,:,idxs] = vals + torch.randn_like(vals)*sigma
 return o
 return out
 return hook

# Registrar
handles = []
for layer_idx, layer in enumerate(get_encoder_layers(model)):
 local = [i - layer_idx*model.config.hidden_size for i in top_neurons
 if layer_idx*model.config.hidden_size <= i < (layer_idx+1)*model.config.hidden_size]
 if not local: continue
 handles.append(
 layer.output.register_forward_hook(make_partial_noise_hook(local, SIGMA))
 )
'''

def make_partial_noise_hook(indices, sigma):
 """
 Inject Gaussian noise only into the given neuron indices,
 robust for:
 - Tensor outputs
 - Tuple outputs (e.g., DistilBERT)
 """
 idxs = torch.tensor(indices, dtype=torch.long)

 def hook(module, inp, out):
 # DistilBERT and some others return tuples: (hidden_state, ...)
 if isinstance(out, tuple):
 main = out[0]
 else:
 main = out

 if not torch.is_tensor(main):
 return out

 if main.dim() == 3 and idxs.numel() > 0:
 # main: [B, seq_len, hidden_dim]
 new_main = main.clone()
 vals = new_main[:, :, idxs]
 new_main[:, :, idxs] = vals + torch.randn_like(vals) * sigma

 # Reconstruct tuple if needed
 if isinstance(out, tuple):
 return (new_main,) + out[1:]

 return new_main

 return out

 return hook

handles = []
encoder_layers = get_encoder_layers(model)
hidden = model.config.hidden_size

for layer_idx, layer in enumerate(encoder_layers):
 local = [
 i - layer_idx * hidden
 for i in top_neurons
 if layer_idx * hidden <= i < (layer_idx + 1) * hidden
 ]
 if not local:
 continue

 # BERT / Longformer have `.output`
 if hasattr(layer, "output"):
 h = layer.output.register_forward_hook(
 make_partial_noise_hook(local, SIGMA)
 )
 # DistilBERT hook entire block
 else:
 h = layer.register_forward_hook(
 make_partial_noise_hook(local, SIGMA)
 )

 handles.append(h)
 
# Inferencia
model.eval()
predictions_gnoise = []
for row in sample_df.itertuples():
 input_ids = torch.tensor(row.input_ids).unsqueeze(0).to(model.device)
 att_mask = torch.tensor(row.attention_mask).unsqueeze(0).to(model.device)
 with torch.no_grad():
 logits = model(input_ids=input_ids, attention_mask=att_mask).logits
 predictions_gnoise.append(int(logits.argmax(dim=-1)))

# ===============================
# Evaluation
# ===============================
accuracy = accuracy_score(labels_list, predictions_gnoise)
f1 = f1_score(labels_list, predictions_gnoise, average='weighted')
report_dict = classification_report(labels_list, predictions_gnoise, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose().round(4).drop("accuracy", errors="ignore")

accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [report_df["support"].sum()]
}, index=["overall_accuracy"])
final_df = pd.concat([report_df, accuracy_row])

# ===============================
# Save report
# ===============================
experiment_title = f"Partial Global Noise ({PERCENTAGE:.0%} top neurons, σ={SIGMA})"
os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
mode = "w"if not os.path.exists(REPORT_PATH) else "a"
with open(REPORT_PATH, mode) as f:
 if mode == "w":
 f.write(f"# {experiment_title}\n")
 else:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(f)

logger.info(f"Accuracy under Partial Global Noise: {accuracy:.4f}")
logger.info(f"Weighted F1 Score: {f1:.4f}")
logger.info(f"Classification report saved to {REPORT_PATH}")

# Cleanup
for h in handles:
 h.remove()

## Fault Sneaking (sim)

In [ ]:
import torch
import os, json, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

def find_final_linear_layer(model):
 """
 Find the final Linear layer used for classification.
 Works for BERT, DistilBERT, BigBird, Longformer, etc.
 It selects the Linear whose out_features == config.num_labels.
 """
 if not hasattr(model, "classifier"):
 raise ValueError("Model has no 'classifier' attribute.")

 clf = model.classifier
 num_labels = model.config.num_labels

 # Case 1: classifier is directly a Linear with num_labels outputs
 if hasattr(clf, "weight") and getattr(clf.weight, "shape", None) is not None:
 if clf.weight.shape[0] == num_labels:
 return clf

 # Case 2: classifier is a head module (e.g. BigBird/Longformer)
 for _, module in clf.named_modules():
 if hasattr(module, "weight") and getattr(module.weight, "shape", None) is not None:
 if module.weight.shape[0] == num_labels:
 return module

 raise ValueError("Could not find a final Linear layer with num_labels outputs inside classifier.")

In [ ]:
'''

import os, json, torch, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Parámetros
PERCENTAGE = 0.10
DELTA_SCALE = 0.05
REPORT_PATH = os.path.join(BASE_PATH, "results", f"fault_sneak_{int(PERCENTAGE*100)}p.csv")

# 1) Selección top‐k
top_neurons = get_top_k_neurons_exact(probe, percentage=PERCENTAGE)
os.makedirs(f"{BASE_PATH}/neurons", exist_ok=True)
with open(f"{BASE_PATH}/neurons/top_{int(PERCENTAGE*100)}p_fault.json", "w") as f:
 json.dump(top_neurons, f, indent=2)

# 2) Detectar la capa final correctamente (universal)
if hasattr(model, "classifier"):
 out_proj = model.classifier
else:
 raise ValueError("No se encontró la capa final 'classifier' en el modelo.")

hidden = model.config.hidden_size

# Crear delta del mismo tamaño
delta = torch.zeros_like(out_proj.weight.data)

# Añadir ruido por cada neurona importante
for g in top_neurons:
 idx = g % hidden
 delta[:, idx] = torch.randn(delta.shape[0]) * DELTA_SCALE

# 3) Hook pre-forward
def make_fault_hook(delta_tensor):
 def hook(module, inp):
 module.weight.data += delta_tensor.to(module.weight.device)
 return hook

handle = out_proj.register_forward_pre_hook(make_fault_hook(delta))

# 4) Inferencia
model.eval()
predictions_fault = []
for row in sample_df.itertuples():
 input_ids = torch.tensor(row.input_ids).unsqueeze(0).to(model.device)
 att_mask = torch.tensor(row.attention_mask).unsqueeze(0).to(model.device)
 with torch.no_grad():
 outputs = model(input_ids=input_ids, attention_mask=att_mask)
 logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]
 predictions_fault.append(int(logits.argmax(dim=-1)))

# ===============================
# Evaluation
# ===============================
accuracy = accuracy_score(labels_list, predictions_fault)
f1 = f1_score(labels_list, predictions_fault, average='weighted')
report_dict = classification_report(labels_list, predictions_fault, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose().round(4).drop("accuracy", errors="ignore")

accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [report_df["support"].sum()]
}, index=["overall_accuracy"])
final_df = pd.concat([report_df, accuracy_row])

# ===============================
# Save report
# ===============================
experiment_title = f"Fault Sneaking ({PERCENTAGE:.0%} top neurons, scale={DELTA_SCALE})"
os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
mode = "w"if not os.path.exists(REPORT_PATH) else "a"
with open(REPORT_PATH, mode) as f:
 if mode == "w":
 f.write(f"# {experiment_title}\n")
 else:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(f)

logger.info(f"Accuracy under Fault Sneaking: {accuracy:.4f}")
logger.info(f"Weighted F1 Score: {f1:.4f}")
logger.info(f"Classification report saved to {REPORT_PATH}")

# 5) Cleanup
handle.remove()
'''

In [ ]:
def run_fault_sneaking_attack(
 model,
 sample_df,
 labels_list,
 probe,
 percentage: float = 0.10,
 delta_scale: float = 0.05,
 base_path: str = BASE_PATH,
 report_name: str = None,
 experiment_title: str = None,
 logger=None,
):
 """
 Fault Sneaking attack:
 - Selects top-k important neurons (global) using the probe.
 - Adds a small random perturbation to the classification weights
 at the positions associated with these neurons.
 - Evaluates the model under the modified classifier weights.
 - Restores the original weights at the end.
 """

 if logger is None:
 import logging
 logger = logging.getLogger(__name__)

 # -------------------------
 # Parameters and paths
 # -------------------------
 perc_int = int(percentage * 100)
 if report_name is None:
 report_name = f"fault_sneak_{perc_int}p.csv"
 report_path = os.path.join(base_path, "results", report_name)

 # -------------------------
 # 1) Select top-k neurons
 # -------------------------
 top_neurons = get_top_k_neurons_exact(probe, percentage=percentage)
 logger.info(f"[FaultSneak] Using top {perc_int}% neurons (count={len(top_neurons)})")

 neurons_dir = os.path.join(base_path, "neurons")
 os.makedirs(neurons_dir, exist_ok=True)
 json_path = os.path.join(neurons_dir, f"top_{perc_int}p_fault.json")
 with open(json_path, "w") as f:
 json.dump(top_neurons, f, indent=2)
 logger.info(f"[FaultSneak] Saved neuron indices to {json_path}")

 # -------------------------
 # 2) Find final classification layer
 # -------------------------
 out_proj = find_final_linear_layer(model)
 hidden = model.config.hidden_size

 # Backup original weights
 orig_weight = out_proj.weight.data.clone()

 # -------------------------
 # 3) Build delta
 # -------------------------
 delta = torch.zeros_like(orig_weight)
 for g in top_neurons:
 idx = g % hidden
 delta[:, idx] = torch.randn(delta.shape[0]) * delta_scale

 logger.info(
 f"[FaultSneak] Injecting weight noise at {len(top_neurons)} neuron positions "
 f"(percentage={percentage:.2%}, scale={delta_scale})"
 )

 # Apply the perturbation once (hardware-like fault) 
 out_proj.weight.data = orig_weight + delta.to(orig_weight.device)

 # -------------------------
 # 4) Inference under attack
 # -------------------------
 model.eval()
 predictions_fault = []

 for row in sample_df.itertuples():
 input_ids = torch.tensor(row.input_ids).unsqueeze(0).to(model.device)
 att_mask = torch.tensor(row.attention_mask).unsqueeze(0).to(model.device)

 with torch.no_grad():
 outputs = model(input_ids=input_ids, attention_mask=att_mask)
 logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]

 pred = int(logits.argmax(dim=-1))
 predictions_fault.append(pred)

 # -------------------------
 # 5) Evaluation
 # -------------------------
 accuracy = accuracy_score(labels_list, predictions_fault)
 f1 = f1_score(labels_list, predictions_fault, average='weighted')
 report_dict = classification_report(
 labels_list,
 predictions_fault,
 output_dict=True,
 zero_division=0
 )
 report_df = pd.DataFrame(report_dict).transpose().round(4).drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame(
 {
 "precision": [""],
 "recall": [""],
 "f1-score": [accuracy],
 "support": [report_df["support"].sum()],
 },
 index=["overall_accuracy"],
 )
 final_df = pd.concat([report_df, accuracy_row])

 # -------------------------
 # 6) Save report
 # -------------------------
 if experiment_title is None:
 experiment_title = (
 f"Fault Sneaking (top {perc_int}% neurons, scale={delta_scale})"
 )

 os.makedirs(os.path.dirname(report_path), exist_ok=True)
 mode = "w"if not os.path.exists(report_path) else "a"
 with open(report_path, mode) as f:
 if mode == "w":
 f.write(f"# {experiment_title}\n")
 else:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(f)

 logger.info(f"[FaultSneak] Accuracy under attack: {accuracy:.4f}")
 logger.info(f"[FaultSneak] Weighted F1 Score: {f1:.4f}")
 logger.info(f"[FaultSneak] Report saved to {report_path}")

 # -------------------------
 # 7) Restore original weights
 # -------------------------
 out_proj.weight.data = orig_weight
 logger.info("[FaultSneak] Restored original classifier weights")

 return {
 "accuracy": accuracy,
 "f1_weighted": f1,
 "report_path": report_path,
 "num_top_neurons": len(top_neurons),
 "percentage": percentage,
 "delta_scale": delta_scale,
 }

In [ ]:
results_fault = run_fault_sneaking_attack(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 probe=probe,
 percentage=0.10,
 delta_scale=0.05,
 base_path=BASE_PATH,
 logger=logger,
)

results_fault

## Tests

In [ ]:
import torch
from sklearn.metrics import accuracy_score
import logging

logger = logging.getLogger(__name__)

# — Helper de inferencia común —
def run_inference(model, sample_df):
 model.eval()
 preds = []
 for row in sample_df.itertuples():
 input_ids = torch.tensor(row.input_ids).unsqueeze(0).to(model.device)
 att_mask = torch.tensor(row.attention_mask).unsqueeze(0).to(model.device)
 with torch.no_grad():
 logits = model(input_ids=input_ids, attention_mask=att_mask).logits
 preds.append(int(logits.argmax(dim=-1)))
 return preds

# ==============================
# 1) TEST Logit‐Bias Majority Attack (sin cortar ejecución)
# ==============================
def test_logit_bias_majority(
 model,
 sample_df,
 labels_list,
 target_class: int = 3,
 bias: float = 5.0,
 min_frac: float = 0.8
):
 # instalar hook
 def hook(m, inp, out):
 out.logits[:, target_class] += bias
 return out
 handle = model.register_forward_hook(hook)

 # inferencia
 attacked = run_inference(model, sample_df)
 handle.remove()

 # comprobación
 count_target = sum(1 for p in attacked if p == target_class)
 frac = count_target / len(attacked)
 expected = f">= {min_frac:.0%} to class {target_class}"
 actual = f"{frac:.0%} to class {target_class}"
 if frac < min_frac:
 logger.error(f"[LogitBiasMajority] FAILURE: expected {expected}, got {actual}")
 else:
 logger.info(f"[LogitBiasMajority] SUCCESS: {actual} (bias={bias})")

 # métricas
 acc = accuracy_score(labels_list, attacked)
 logger.info(f"[LogitBiasMajority] Accuracy under attack: {acc:.4f}")
 # opcional: devolver attacked para inspección
 return attacked

# ==============================
# 2) TEST Gaussian Noise en top‐k (sin cortar ejecución)
# ==============================
def test_gaussian_noise(
 model,
 sample_df,
 labels_list,
 probe,
 percentage: float = 0.1,
 sigma: float = 0.2
):
 # baseline y atacada
 baseline = run_inference(model, sample_df)
 topk = get_top_k_neurons_exact(probe, percentage=percentage)

 # instalar hooks
 def make_noise(idx_list, σ):
 idxs = torch.tensor(idx_list, dtype=torch.long)
 def hook(m, inp, out):
 main = out[0] if isinstance(out, tuple) else out
 if torch.is_tensor(main) and main.dim()==3 and idxs.numel()>0:
 o = main.clone()
 cls = o[:,0,:]
 cls[:, idxs] += torch.randn_like(cls[:, idxs]) * σ
 o[:,0,:] = cls
 return (o,)+out[1:] if isinstance(out, tuple) else o
 return out
 return hook

 handles = []
 for i, layer in enumerate(get_encoder_layers(model)):
 local = [g - i*model.config.hidden_size for g in topk
 if i*model.config.hidden_size <= g < (i+1)*model.config.hidden_size]
 if not local: continue
 handles.append((layer.output if hasattr(layer, "output") else layer).register_forward_hook(make_noise(local, sigma)))

 attacked = run_inference(model, sample_df)
 for h in handles: h.remove()

 # ver cuántas cambiaron
 diff = sum(1 for b,a in zip(baseline, attacked) if b != a)
 expected = "> 0 changes"
 actual = f"{diff} changes"
 if diff == 0:
 logger.error(f"[GaussNoise] FAILURE: expected {expected}, got {actual}")
 else:
 logger.info(f"[GaussNoise] SUCCESS: {actual} (σ={sigma}, {percentage:.0%} neurons)")

 acc = accuracy_score(labels_list, attacked)
 logger.info(f"[GaussNoise] Accuracy under attack: {acc:.4f}")
 return attacked

# ==============================
# 3) TEST Partial Global Noise en top‐k (sin cortar ejecución)
# ==============================
def test_partial_noise(
 model,
 sample_df,
 labels_list,
 probe,
 percentage: float = 0.1,
 sigma: float = 0.1
):
 baseline = run_inference(model, sample_df)
 topk = get_top_k_neurons_exact(probe, percentage=percentage)

 def make_partial(idx_list, σ):
 idxs = torch.tensor(idx_list, dtype=torch.long)
 def hook(m, inp, out):
 main = out[0] if isinstance(out, tuple) else out
 if torch.is_tensor(main) and main.dim()==3 and idxs.numel()>0:
 o = main.clone()
 o[:,:,idxs] += torch.randn_like(o[:,:,idxs]) * σ
 return (o,)+out[1:] if isinstance(out, tuple) else o
 return out
 return hook

 handles = []
 for i, layer in enumerate(get_encoder_layers(model)):
 local = [g - i*model.config.hidden_size for g in topk
 if i*model.config.hidden_size <= g < (i+1)*model.config.hidden_size]
 if not local: continue
 handles.append((layer.output if hasattr(layer, "output") else layer).register_forward_hook(make_partial(local, sigma)))

 attacked = run_inference(model, sample_df)
 for h in handles: h.remove()

 diff = sum(1 for b,a in zip(baseline, attacked) if b != a)
 expected = "> 0 changes"
 actual = f"{diff} changes"
 if diff == 0:
 logger.error(f"[PartialNoise] FAILURE: expected {expected}, got {actual}")
 else:
 logger.info(f"[PartialNoise] SUCCESS: {actual} (σ={sigma}, {percentage:.0%} neurons)")

 acc = accuracy_score(labels_list, attacked)
 logger.info(f"[PartialNoise] Accuracy under attack: {acc:.4f}")
 return attacked

# ==============================
# 4) TEST Fault‐Sneaking simulado en top‐k (sin cortar ejecución)
# ==============================
def test_fault_sneaking(
 model,
 sample_df,
 labels_list,
 probe,
 percentage: float = 0.1,
 delta_scale: float = 0.05
):
 baseline = run_inference(model, sample_df)
 topk = get_top_k_neurons_exact(probe, percentage=percentage)

 # delta sobre out_proj
 hidden = model.config.hidden_size
 out_proj = find_final_linear_layer(model) # arch-agnostic (BERT Linear / BigBird head / DistilBERT)
 delta = torch.zeros_like(out_proj.weight.data)
 for g in topk:
 idx = g % hidden
 delta[:, idx] = torch.randn(delta.shape[0]) * delta_scale

 # hook
 def make_fault_hook(delta_tensor):
 def hook(m, inp):
 m.weight.data += delta_tensor.to(m.weight.device)
 return hook

 handle = out_proj.register_forward_pre_hook(make_fault_hook(delta))
 attacked = run_inference(model, sample_df)
 handle.remove()

 diff = sum(1 for b,a in zip(baseline, attacked) if b != a)
 expected = "> 0 changes"
 actual = f"{diff} changes"
 if diff == 0:
 logger.error(f"[FaultSneak] FAILURE: expected {expected}, got {actual}")
 else:
 logger.info(f"[FaultSneak] SUCCESS: {actual} (scale={delta_scale}, {percentage:.0%} neurons)")

 acc = accuracy_score(labels_list, attacked)
 logger.info(f"[FaultSneak] Accuracy under attack: {acc:.4f}")
 return attacked

# ==============================
# Ejecutar tests sin cortar ejecución
# ==============================
try:
 att_logit = test_logit_bias_majority(model, sample_df, labels_list)
 att_gauss = test_gaussian_noise(model, sample_df, labels_list, probe)
 att_partial = test_partial_noise(model, sample_df, labels_list, probe)
 att_fault = test_fault_sneaking(model, sample_df, labels_list, probe)
 logger.info("All attack tests completed.")
except Exception as _e:
 logger.warning(f"[legacy tests] skipped (non-critical): {type(_e).__name__}: {_e}")

# Conjuntos disjuntos de neuronas

In [ ]:
from collections import defaultdict
import os
import json

top_percentage = 0.1 # 50% 

# Detect and convert (layer, neuron) tuples to global indices if needed
hidden_dim = model.config.hidden_size

def tuple_to_global_index(neuron_tuples, hidden_dim):
 return [layer * hidden_dim + neuron for (layer, neuron) in neuron_tuples]

per_class_top_indices = {}
for class_id, neuron_list in per_class_top_neurons.items():
 if len(neuron_list) > 0 and isinstance(neuron_list[0], tuple):
 per_class_top_indices[class_id] = tuple_to_global_index(neuron_list, hidden_dim)
 else:
 per_class_top_indices[class_id] = neuron_list

# Exclusive neurons: present in top of class A but not in any other class
exclusive_class_neurons = {}
for cid, own_top in per_class_top_indices.items():
 other = set()
 for other_cid, other_top in per_class_top_indices.items():
 if other_cid != cid:
 other.update(other_top)
 exclusive = sorted(set(own_top) - other)
 exclusive_class_neurons[cid] = exclusive
 print(f"Class {cid}: {len(exclusive)} exclusive neurons out of {len(own_top)} top neurons")

# Save exclusive neurons to JSON
exclusive_dir = f"{BASE_PATH}/exclusive_neurons"
os.makedirs(exclusive_dir, exist_ok=True)
for class_id, neuron_list in exclusive_class_neurons.items():
 path = f"{exclusive_dir}/exclusive_top{int(top_percentage*100)}p_class_{class_id}.json"
 with open(path, "w") as f:
 json.dump([int(x) for x in neuron_list], f, indent=2)
 logger.info(f"Saved exclusive neurons for class {class_id} to {path}")

In [ ]:
def silence_exclusive_class_and_evaluate(
 model,
 sample_df,
 labels_list,
 exclusive_neuron_indices,
 class_id,
 report_path=None,
 experiment_title=None
):
 hidden_dim = model.config.hidden_size
 num_layers = model.config.num_hidden_layers

 logger.info(f"Silencing {len(exclusive_neuron_indices)} EXCLUSIVE neurons for class {class_id}")

 encoder_layers = get_encoder_layers(model)
 hook_handles = []
 for i in range(num_layers):
 indices_layer = [idx - i * hidden_dim for idx in exclusive_neuron_indices if i * hidden_dim <= idx < (i + 1) * hidden_dim]
 if indices_layer:
 logger.info(f"Layer {i}: silencing {len(indices_layer)} exclusive neurons for class {class_id}")
 if hasattr(encoder_layers[i], "output"):
 handle = encoder_layers[i].output.register_forward_hook(make_cls_silence_hook(indices_layer))
 else:
 handle = encoder_layers[i].register_forward_hook(make_cls_silence_hook(indices_layer))
 hook_handles.append(handle)

 # --- Evaluation ---
 model.eval()
 predictions = []
 for i in range(len(sample_df)):
 input_ids_tensor = torch.tensor(sample_df.loc[i, 'input_ids']).unsqueeze(0).to(model.device)
 attention_mask_tensor = torch.tensor(sample_df.loc[i, 'attention_mask']).unsqueeze(0).to(model.device)
 with torch.no_grad():
 outputs = model(input_ids=input_ids_tensor, attention_mask=attention_mask_tensor)
 logits = outputs['logits']
 pred = torch.argmax(logits, dim=1).item()
 predictions.append(pred)
 del input_ids_tensor, attention_mask_tensor, outputs, logits
 torch.cuda.empty_cache()

 # --- Metrics & reporting ---
 from sklearn.metrics import accuracy_score, f1_score, classification_report
 import pandas as pd
 accuracy = accuracy_score(labels_list, predictions)
 f1 = f1_score(labels_list, predictions, average='weighted')
 report_dict = classification_report(labels_list, predictions, output_dict=True)
 report_df = pd.DataFrame(report_dict).transpose().round(4)
 report_df = report_df.drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 # --- Save classification report ---
 if report_path is None:
 report_path = f"{BASE_PATH}/results/exclusive_class_silencing_{class_id}.csv"
 os.makedirs(os.path.dirname(report_path), exist_ok=True)
 if experiment_title is None:
 experiment_title = f"Silencing exclusive neurons for class {class_id}"
 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Accuracy after exclusive silencing: {accuracy:.4f}")
 logger.info(f"Weighted F1 Score: {f1:.4f}")
 logger.info(f"Classification report saved to {report_path}")

 # Remove hooks
 for handle in hook_handles:
 handle.remove()
 logger.info("All hooks removed after evaluation")

In [ ]:
# --- Run silencing experiments for each class with its exclusive neurons ---
for class_id, neuron_indices in exclusive_class_neurons.items():
 silence_exclusive_class_and_evaluate(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 exclusive_neuron_indices=neuron_indices,
 class_id=class_id,
 report_path=f"{BASE_PATH}/results/exclusive_class_silencing_{class_id}.csv",
 experiment_title=f"Silencing exclusive neurons for class {class_id}"
 )

# Detection metrics (normal-vs-malicious) — R3.2

Adds the detection-oriented lens requested by Reviewer R3.2 on the malware model: from the malicious score (1 − P(Normal)) it reports EER, TPR@1%FPR, FAR, MAR and ROC-AUC for the baseline and global silencing. Uses the module `detection_metrics.py`. macro-F1 stays the primary metric; not applied to GoEmotions.

In [ ]:
# === Detection metrics (R3.2) + silencing macro-F1 — MULTI-SEED (R4.4/R7.3) ===
# Reports mean±std over N_SEEDS, each seed RE-SAMPLING the eval set (so std>0; per the
# decision, base models are deterministic -> a naive re-run gives std≈0). Covers baseline
# + global silencing at each SWEEP_PCT. Detection lens: malicious_score = 1 - P(Normal),
# Normal = 3. Defines eval_probs / silence hooks / _make_eval_sample (reused below).
import os, ast, numpy as np, pandas as pd, torch
from detection_metrics import detection_metrics
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

# --- Ensure a CLEAN model before the new experiments ---
# Legacy attack cells (e.g. inline Fault-Sneaking) mutate classifier weights in place via a
# forward hook and DON'T restore them -> reload the checkpoint so every new-experiment eval
# starts from the true model.
model.load_state_dict(torch.load(weights_path, map_location=model.device), strict=False)
model.eval()

def eval_probs(model, sample_df):
 model.eval(); dev = model.device; out = []
 for row in sample_df.itertuples():
 ii = torch.tensor(row.input_ids).unsqueeze(0).to(dev)
 am = torch.tensor(row.attention_mask).unsqueeze(0).to(dev)
 with torch.no_grad():
 lg = model(input_ids=ii, attention_mask=am).logits
 out.append(torch.softmax(lg, -1).squeeze(0).float().cpu().numpy())
 return np.vstack(out)

def _register_silence_hooks(model, probe, percentage):
 hd = model.config.hidden_size; nl = model.config.num_hidden_layers
 top = get_top_k_neurons_exact(probe, percentage=percentage)
 layers = get_encoder_layers(model); handles = []
 for i in range(nl):
 idxs = [idx - i * hd for idx in top if i * hd <= idx < (i + 1) * hd]
 if idxs:
 tgt = layers[i].output if hasattr(layers[i], "output") else layers[i]
 handles.append(tgt.register_forward_hook(make_cls_silence_hook(idxs)))
 return handles

# --- Held-out evaluation pool (transferability protocol) ---
# Evaluate the model/interventions ONLY on examples the probe did NOT train on: take the
# COMPLEMENT of the probe's training split (same reduction_ratio + random_state=42 as the
# reduction cell). This tests the model on the input CSV examples, never on probe data.
from sklearn.model_selection import train_test_split as _tts
_full_df = pd.read_csv(input_csv)
_full_df['label'] = LABEL_ENCODER.transform(_full_df['label'].astype(str))
try:
 _, _EVAL_POOL = _tts(_full_df, train_size=reduction_ratio, stratify=_full_df['label'], random_state=42)
except Exception as _e:
 print(f"[R3.2] held-out split fallback (using full CSV): {_e}")
 _EVAL_POOL = _full_df
_EVAL_POOL = _EVAL_POOL.reset_index(drop=True)
print(f"[R3.2] held-out eval pool: {len(_EVAL_POOL)} examples (probe trained on the other {len(_full_df)-len(_EVAL_POOL)})")

def _make_eval_sample(n, seed):
 s = _EVAL_POOL.sample(n=min(n, len(_EVAL_POOL)), random_state=seed).reset_index(drop=True)
 s['input_ids'] = s['input_ids'].apply(ast.literal_eval)
 s['attention_mask'] = s['attention_mask'].apply(ast.literal_eval)
 return s, np.asarray(s['label'].tolist(), dtype=int)

if not RUN_DETECTION_METRICS:
 print("[R3.2] skipped (RUN_DETECTION_METRICS=False)")
else:
 _acc = {}
 for _seed in range(N_SEEDS):
 sdf, yt = _make_eval_sample(SAMPLE_N, _seed)
 def _rec(cond, probs):
 m = detection_metrics(probs, yt, normal_idx=NORMAL_IDX)
 m["macro_f1"] = f1_score(yt, probs.argmax(1), average="macro", zero_division=0)
 _acc.setdefault(cond, []).append(m)
 _rec("baseline", eval_probs(model, sdf))
 for pct in SWEEP_PCTS:
 h = _register_silence_hooks(model, probe, pct)
 try:
 probs = eval_probs(model, sdf)
 finally:
 for hh in h: hh.remove()
 _rec(f"silencing_{int(pct*100)}p", probs)
 print(f"[R3.2] seed {_seed+1}/{N_SEEDS} done")
 keys = ["macro_f1", "roc_auc", "eer", "tpr_at_fpr", "far_argmax", "mar_argmax"]
 rows = []
 for cond, lst in _acc.items():
 r = {"condition": cond, "n_seeds": N_SEEDS}
 for k in keys:
 v = np.array([d[k] for d in lst], float)
 r[f"{k}_mean"] = float(np.nanmean(v))
 r[f"{k}_std"] = float(np.nanstd(v, ddof=1)) if len(v) > 1 else 0.0
 rows.append(r)
 os.makedirs(f"{BASE_PATH}/results", exist_ok=True)
 pd.DataFrame(rows).to_csv(f"{BASE_PATH}/results/detection_and_f1_malware.csv", index=False)
 print(f"[R3.2] saved mean±std over {N_SEEDS} seeds -> detection_and_f1_malware.csv")


## Random-neuron control + significance — R1.3 / R4.2

Control requested by Reviewers R1.3 and R4.2 (correlation vs. causation): silencing the top-k ranked neurons must degrade macro-F1 more than silencing k random neurons. Reports the drop, the random distribution (mean +/- std over seeds), a one-sided empirical p-value and a z-score. Uses `perturbation_stats.py`. Also provides the seed-aggregation helper for R4.4/R7.3.

In [ ]:
# === Random-neuron control + significance — R1.3 / R4.2 ===
# Silencing the top-k ranked neurons must hurt macro-F1 more than k random neurons. The
# RANDOM_CONTROL_DRAWS random draws provide the null distribution / p-value (that IS the
# statistical validation here, complementing the multi-seed detection/F1 cell above).
import perturbation_stats
from sklearn.metrics import f1_score
if not RUN_RANDOM_CONTROL:
 print("[R1.3] skipped (RUN_RANDOM_CONTROL=False)")
else:
 _sdf, _yt = _make_eval_sample(SAMPLE_N, 0)
 def _hooks_for(indices):
 hd = model.config.hidden_size; nl = model.config.num_hidden_layers
 layers = get_encoder_layers(model); handles = []
 for i in range(nl):
 idxs = [idx - i * hd for idx in indices if i * hd <= idx < (i + 1) * hd]
 if idxs:
 tgt = layers[i].output if hasattr(layers[i], "output") else layers[i]
 handles.append(tgt.register_forward_hook(make_cls_silence_hook(idxs)))
 return handles
 def _f1(indices):
 h = _hooks_for(indices)
 try:
 p = eval_probs(model, _sdf)
 finally:
 for hh in h: hh.remove()
 return f1_score(_yt, p.argmax(1), average="macro", zero_division=0)
 total = model.config.hidden_size * model.config.num_hidden_layers
 base = _f1([]); rows = []
 for pct in SWEEP_PCTS:
 r = perturbation_stats.top_vs_random(_f1, base, get_top_k_neurons_exact(probe, percentage=pct),
 total, n_seeds=RANDOM_CONTROL_DRAWS, base_seed=0, higher_is_better=True)
 r["percentage"] = pct; rows.append(r)
 cols = ["percentage","k","baseline","top_metric","top_drop","random_mean_metric",
 "random_mean_drop","random_std_drop","p_empirical","z_score","n_seeds"]
 pd.DataFrame([{c: r[c] for c in cols} for r in rows]).to_csv(f"{BASE_PATH}/results/random_control_malware.csv", index=False)
 print("[R1.3] saved")


## [CLS] vs mean-pool — malware validation only (full ablation in GoEmotions) — R4.3 / R3.4 / R7.6


In [ ]:
# === Mean-pool VALIDATION on malware (single example) — R4.3 / R3.4 / R7.6 ===
# Per the decision (option c, "one or few representative models"), the FULL
# [CLS]-vs-mean-pool ranking agreement is reported on GoEmotions (short sequences).
# On malware, syscall token sequences are very long, so here we only VALIDATE that the
# mean-pooled extraction path works on a SINGLE example; the paper justifies restricting
# the full ablation to GoEmotions by this sequence length. Gated by RUN_MEANPOOL_ABLATION.
if not RUN_MEANPOOL_ABLATION:
 print("[R4.3] mean-pool malware validation skipped (RUN_MEANPOOL_ABLATION=False)")
else:
 _one = df_reduced["input_ids"].tolist()[:1]
 _val_actf = f"{BASE_PATH}/activations_mean_validation.json"
 transformers_extractor.extract_representations(
 model, _one, _val_actf, device=model.device, pooling="mean",
 )
 _acts_val, _nl = data_loader.load_activations(_val_actf)
 print(f"[R4.3] mean-pool extraction VALIDATED on malware ({MODEL}): 1 example, "
 f"{_nl} layers. Full [CLS]-vs-mean-pool agreement is reported on GoEmotions "
 f"(short sequences); malware syscall sequences are too long for the full ablation.")


## Attribution-guided bit-flip attack — R7.9 (hardware faults / tampering, TH-3)


In [ ]:
# === Attribution-guided bit-flip (R7.9) — MULTI-SEED mean±std ===
import bitflip_attack
from sklearn.metrics import f1_score
if not RUN_BITFLIP:
 print("[R7.9] skipped (RUN_BITFLIP=False)")
else:
 out_proj = find_final_linear_layer(model); hidden = model.config.hidden_size
 orig = out_proj.weight.data.clone()
 _acc = {}
 for _seed in range(N_SEEDS):
 sdf, yt = _make_eval_sample(SAMPLE_N, _seed)
 def _f1():
 p = eval_probs(model, sdf)
 return f1_score(yt, np.nan_to_num(p, nan=-np.inf).argmax(1), average="macro", zero_division=0)
 _acc.setdefault("baseline", []).append(_f1())
 for pct in SWEEP_PCTS:
 cols = sorted({int(g) % hidden for g in get_top_k_neurons_exact(probe, percentage=pct)})
 try:
 bitflip_attack.flip_exponent_msb_columns_(out_proj.weight.data, cols); f1 = _f1()
 finally:
 out_proj.weight.data.copy_(orig)
 _acc.setdefault(f"bitflip_{int(pct*100)}p", []).append(f1)
 rows = []
 for cond, vals in _acc.items():
 v = np.array(vals, float)
 rows.append({"condition": cond, "n_seeds": N_SEEDS,
 "f1_mean": float(v.mean()), "f1_std": float(v.std(ddof=1)) if len(v) > 1 else 0.0})
 pd.DataFrame(rows).to_csv(f"{BASE_PATH}/results/bitflip_malware.csv", index=False)
 print(f"[R7.9] saved mean±std over {N_SEEDS} seeds")


## Alternative attribution rankings vs probe — #8 (R1.1 / R3.3 / R7.5)


In [ ]:
# === Alternative attribution rankings vs probe (#8 — R1.1 / R3.3 / R7.5) ===
# Compares SYNAPSE's linear-probe ranking against two neuron-level attribution methods
# (activation x gradient, and conductance = integrated-gradients-based) via rank
# correlation + top-k overlap. The probe stays operational; this only validates it.
# Conductance (captum) is HEAVY -> run on a bounded subset (ATTRIBUTION_N_SAMPLES).
import attribution_methods, ranking_agreement, pandas as pd

if not RUN_ATTRIBUTION:
 print("[#8] skipped (RUN_ATTRIBUTION=False)")
else:
 _enc = get_encoder_layers(model)
 _layer_mods = [(l.output if hasattr(l, "output") else l) for l in _enc]
 _sub = sample_df.head(ATTRIBUTION_N_SAMPLES)
 _dev = model.device
 _samples = list(_sub.itertuples())
 imp_probe = ranking_agreement.probe_importance(probe)

 def _fwd_row(row):
 ii = torch.tensor(row.input_ids).unsqueeze(0).to(_dev)
 am = torch.tensor(row.attention_mask).unsqueeze(0).to(_dev)
 return model(input_ids=ii, attention_mask=am).logits

 rows = []
 # activation x gradient (cheap, pure torch)
 imp_ag = attribution_methods.activation_times_gradient(_fwd_row, _samples, _layer_mods,
 target="pred", cls_pos=0)
 agr_ag = ranking_agreement.compare_rankings(imp_probe, imp_ag, top_k_frac=0.10)
 print("[#8] probe vs activation x gradient:", agr_ag)
 rows.append({"method": "activation_times_gradient", **agr_ag})

 # conductance (captum, heavy) — guarded so a failure can't abort the run
 try:
 imp_cond = attribution_methods.conductance_importance(_fwd_row, _samples, _layer_mods,
 target="pred", cls_pos=0,
 n_steps=ATTRIBUTION_STEPS)
 agr_cond = ranking_agreement.compare_rankings(imp_probe, imp_cond, top_k_frac=0.10)
 print("[#8] probe vs conductance:", agr_cond)
 rows.append({"method": "conductance", **agr_cond})
 except Exception as e:
 print(f"[#8][WARN] conductance failed ({type(e).__name__}: {e}); reporting act x grad only")

 attr_df = pd.DataFrame(rows)
 os.makedirs(f"{BASE_PATH}/results", exist_ok=True)
 attr_out = f"{BASE_PATH}/results/attribution_agreement_malware.csv"
 attr_df.to_csv(attr_out, index=False)
 print("[#8] saved ->", attr_out)


In [ ]:
# === Save profiling results (R7.8 complexity: time + memory) ===
_prof_path = f"{BASE_PATH}/results/complexity_time_memory.csv"if "BASE_PATH"in globals() else "results/complexity_time_memory.csv"
print(f"[profile] TOTAL notebook wall-clock so far: {time.perf_counter() - PROFILE_T0:.1f}s")
PROFILE.save(_prof_path)
